# CIFAR Random Test Infusion - Results Analysis Notebook

This notebook analyzes results from the random test infusion experiment.
It can be run while the experiment is running to monitor progress.

In [1]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from glob import glob
from collections import defaultdict
import pandas as pd
import seaborn as sns

## Configuration

In [ ]:
RESULTS_DIR = './results/random_test_infusion/'
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Analyzing results from: {RESULTS_DIR}")

## Load Experiment Metadata

In [3]:
metadata_path = os.path.join(RESULTS_DIR, 'experiment_metadata.json')
if os.path.exists(metadata_path):
    with open(metadata_path, 'r') as f:
        exp_metadata = json.load(f)
    print("Experiment Metadata:")
    for key, value in exp_metadata.items():
        print(f"  {key}: {value}")
else:
    print("No experiment metadata found yet.")
    exp_metadata = {}

## Load Experiment Log

In [4]:
log_path = os.path.join(RESULTS_DIR, 'experiment_log.jsonl')
log_entries = []

if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        for line in f:
            log_entries.append(json.loads(line.strip()))

    print(f"\nLoaded {len(log_entries)} experiment log entries")

    # Convert to DataFrame for easier analysis
    df_log = pd.DataFrame(log_entries)
    print(f"\nColumns: {list(df_log.columns)}")
    print(f"Shape: {df_log.shape}")
else:
    print("No experiment log found yet.")
    df_log = pd.DataFrame()



## Summary Statistics

In [5]:
if len(df_log) > 0:
    print("\n" + "="*80)
    print("SUMMARY STATISTICS")
    print("="*80)

    n_completed = len(df_log)
    n_samples = df_log['sample_idx'].nunique()
    n_classes = 10
    expected_total = exp_metadata.get('n_samples', 1000) * n_classes

    print(f"\nProgress: {n_completed}/{expected_total} ({n_completed/expected_total*100:.1f}%)")
    print(f"Unique test images processed: {n_samples}/{exp_metadata.get('n_samples', 'unknown')}")

    print(f"\nΔp statistics (change in target probability):")
    print(f"  Mean: {df_log['delta_prob'].mean():+.6f}")
    print(f"  Std: {df_log['delta_prob'].std():.6f}")
    print(f"  Min: {df_log['delta_prob'].min():+.6f}")
    print(f"  Max: {df_log['delta_prob'].max():+.6f}")
    print(f"  Median: {df_log['delta_prob'].median():+.6f}")

    # Success rate (positive Δp)
    n_success = (df_log['delta_prob'] > 0).sum()
    success_rate = n_success / len(df_log) * 100
    print(f"\nSuccess rate (Δp > 0): {n_success}/{len(df_log)} ({success_rate:.1f}%)")

    # By target class
    print("\nΔp by target class:")
    for target_class in range(10):
        class_data = df_log[df_log['target_class'] == target_class]
        if len(class_data) > 0:
            mean_delta = class_data['delta_prob'].mean()
            n_pos = (class_data['delta_prob'] > 0).sum()
            print(f"  {CLASS_NAMES[target_class]:>12}: mean Δp = {mean_delta:+.6f}, success = {n_pos}/{len(class_data)} ({n_pos/len(class_data)*100:.1f}%)")

## Visualizations

In [ ]:
def load_experiment_result(exp_dir):
    """Load a single experiment result"""
    result_path = os.path.join(exp_dir, 'results.npz')
    if os.path.exists(result_path):
        return np.load(result_path, allow_pickle=True)
    return None


In [7]:
if len(df_log) > 0:
    print("\n" + "="*80)
    print("GENERATING VISUALIZATIONS")
    print("="*80)

    # First, we need to load the logits from the results to compute which class has highest probability
    # We'll compute success as: argmax(prob_infused) == target_class
    
    # Load all experiment results to compute success metric
    success_list = []
    for idx, row in df_log.iterrows():
        exp_dir = os.path.join(RESULTS_DIR, f"sample_{row['sample_idx']:04d}_test_{row['test_image_idx']}_target_{row['target_class']}")
        result = load_experiment_result(exp_dir)
        if result is not None:
            logits_infused = result['logits_infused'][0]
            probs_infused = np.exp(logits_infused - np.max(logits_infused))
            probs_infused /= probs_infused.sum()
            predicted_class = np.argmax(probs_infused)
            is_success = (predicted_class == row['target_class'])
            success_list.append(is_success)
        else:
            success_list.append(False)
    
    df_log['success'] = success_list
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    # 1. Distribution of Δp
    ax1 = axes[0, 0]
    ax1.hist(df_log['delta_prob'], bins=50, alpha=0.7, edgecolor='black', color='steelblue')
    ax1.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero change')
    ax1.axvline(df_log['delta_prob'].mean(), color='green', linestyle='--', linewidth=2, label=f'Mean: {df_log["delta_prob"].mean():+.4f}')
    ax1.set_xlabel('Δp (change in target probability)', fontsize=11)
    ax1.set_ylabel('Count', fontsize=11)
    ax1.set_title('Distribution of Δp Across All Experiments', fontsize=12, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # 2. Δp by target class (box plot)
    ax2 = axes[0, 1]
    target_class_data = [df_log[df_log['target_class'] == i]['delta_prob'].values for i in range(10)]
    bp = ax2.boxplot(target_class_data, labels=CLASS_NAMES, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    ax2.axhline(0, color='red', linestyle='--', linewidth=1, alpha=0.5)
    ax2.set_xlabel('Target Class', fontsize=11)
    ax2.set_ylabel('Δp', fontsize=11)
    ax2.set_title('Δp Distribution by Target Class', fontsize=12, fontweight='bold')
    ax2.tick_params(axis='x', rotation=45)
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')
    ax2.grid(True, alpha=0.3, axis='y')

    # 3. Success rate by target class (bar plot)
    # Success = argmax(prob_infused) == target_class
    ax3 = axes[0, 2]
    success_rates = []
    for target_class in range(10):
        class_data = df_log[df_log['target_class'] == target_class]
        if len(class_data) > 0:
            success_rate = class_data['success'].sum() / len(class_data) * 100
            success_rates.append(success_rate)
        else:
            success_rates.append(0)

    bars = ax3.bar(CLASS_NAMES, success_rates, alpha=0.7, edgecolor='black', color='coral')
    ax3.axhline(10, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='10% baseline (random)')
    ax3.set_xlabel('Target Class', fontsize=11)
    ax3.set_ylabel('Success Rate (%)', fontsize=11)
    ax3.set_title('Success Rate (Target = Predicted) by Target Class', fontsize=12, fontweight='bold')
    ax3.tick_params(axis='x', rotation=45)
    plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45, ha='right')
    ax3.grid(True, alpha=0.3, axis='y')
    ax3.legend()

    # Add percentage labels on bars
    for bar in bars:
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%',
                ha='center', va='bottom', fontsize=8)

    # 4. Δp by true label (box plot)
    ax4 = axes[1, 0]
    true_label_data = [df_log[df_log['true_label'] == i]['delta_prob'].values for i in range(10)]
    bp2 = ax4.boxplot(true_label_data, labels=CLASS_NAMES, patch_artist=True)
    for patch in bp2['boxes']:
        patch.set_facecolor('lightgreen')
        patch.set_alpha(0.7)
    ax4.axhline(0, color='red', linestyle='--', linewidth=1, alpha=0.5)
    ax4.set_xlabel('True Label', fontsize=11)
    ax4.set_ylabel('Δp', fontsize=11)
    ax4.set_title('Δp Distribution by True Label', fontsize=12, fontweight='bold')
    ax4.tick_params(axis='x', rotation=45)
    plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45, ha='right')
    ax4.grid(True, alpha=0.3, axis='y')

    # 5. Heatmap: Δp by (true_label, target_class)
    ax5 = axes[1, 1]
    heatmap_data = np.zeros((10, 10))
    counts = np.zeros((10, 10))

    for _, row in df_log.iterrows():
        true_label = int(row['true_label'])
        target_class = int(row['target_class'])
        heatmap_data[true_label, target_class] += row['delta_prob']
        counts[true_label, target_class] += 1

    # Average Δp
    with np.errstate(divide='ignore', invalid='ignore'):
        heatmap_data = np.where(counts > 0, heatmap_data / counts, np.nan)

    im = ax5.imshow(heatmap_data, cmap='RdYlGn', aspect='auto', vmin=-1, vmax=1)
    ax5.set_xticks(range(10))
    ax5.set_yticks(range(10))
    ax5.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
    ax5.set_yticklabels(CLASS_NAMES)
    ax5.set_xlabel('Target Class', fontsize=11)
    ax5.set_ylabel('True Label', fontsize=11)
    ax5.set_title('Mean Δp by (True Label, Target Class)', fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax5, label='Mean Δp')

    # Add text annotations
    for i in range(10):
        for j in range(10):
            if counts[i, j] > 0:
                text = ax5.text(j, i, f'{heatmap_data[i, j]:.3f}',
                               ha="center", va="center", color="black", fontsize=7)

    # 6. Cumulative progress over time
    # Success = argmax(prob_infused) == target_class
    ax6 = axes[1, 2]
    df_log_sorted = df_log.sort_values('timestamp')
    df_log_sorted['cumulative_success_rate'] = df_log_sorted['success'].cumsum() / (np.arange(len(df_log_sorted)) + 1) * 100

    ax6.plot(df_log_sorted['cumulative_success_rate'], linewidth=2, color='blue')
    ax6.axhline(10, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='10% baseline (random)')
    ax6.set_xlabel('Experiment Number', fontsize=11)
    ax6.set_ylabel('Cumulative Success Rate (%)', fontsize=11)
    ax6.set_title('Cumulative Success Rate Over Time\n(Success = Target is Predicted Class)', fontsize=12, fontweight='bold')
    ax6.grid(True, alpha=0.3)
    ax6.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'analysis_summary.png'), dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\nSaved summary plot to: {os.path.join(RESULTS_DIR, 'analysis_summary.png')}")
    
    # Print overall success statistics
    print(f"\n{'='*80}")
    print("SUCCESS STATISTICS (Target = Predicted Class)")
    print(f"{'='*80}")
    n_success = df_log['success'].sum()
    success_rate = n_success / len(df_log) * 100
    print(f"Overall success rate: {n_success}/{len(df_log)} ({success_rate:.1f}%)")
    print(f"Baseline (random): 10%")
    print(f"Improvement over random: {success_rate - 10:.1f}%")

## Load Individual Results

In [8]:

# Find all experiment directories
exp_dirs = sorted(glob(os.path.join(RESULTS_DIR, 'sample_*')))
print(f"\n{len(exp_dirs)} experiment directories found")

if len(exp_dirs) > 0:
    # Load a sample result to demonstrate structure
    print("\nLoading sample result...")
    sample_result = load_experiment_result(exp_dirs[0])

    if sample_result is not None:
        print("\nAvailable keys in result:")
        for key in sample_result.keys():
            value = sample_result[key]
            if isinstance(value, np.ndarray):
                print(f"  {key}: shape = {value.shape}, dtype = {value.dtype}")
            else:
                print(f"  {key}: {type(value)}")

## Visualize Individual Experiments

In [ ]:
def visualize_experiment(exp_dir, result=None):
    """Visualize a single experiment result"""
    if result is None:
        result = load_experiment_result(exp_dir)

    if result is None:
        print(f"Could not load result from {exp_dir}")
        return

    # Extract data
    probe_image = result['probe_image']
    true_label = int(result['true_label'])
    target_class = int(result['target_class'])

    original_train_images = result['original_train_images']
    perturbed_train_images = result['perturbed_train_images']

    logits_epoch10 = result['logits_epoch10'][0]
    logits_infused = result['logits_infused'][0]

    probs_epoch10 = np.exp(logits_epoch10 - np.max(logits_epoch10))
    probs_epoch10 /= probs_epoch10.sum()

    probs_infused = np.exp(logits_infused - np.max(logits_infused))
    probs_infused /= probs_infused.sum()

    delta_probs = probs_infused - probs_epoch10

    # Create visualization
    fig = plt.figure(figsize=(20, 10))
    gs = fig.add_gridspec(3, 11, hspace=0.4, wspace=0.3)

    # Probe image
    ax_probe = fig.add_subplot(gs[:, 0])
    probe_display = np.transpose(probe_image, (1, 2, 0))
    ax_probe.imshow(probe_display)
    ax_probe.set_title(f'Probe Image\nTrue: {CLASS_NAMES[true_label]}\nTarget: {CLASS_NAMES[target_class]}',
                       fontsize=10, fontweight='bold')
    ax_probe.axis('off')

    # Show top 10 training examples
    n_show = min(10, len(original_train_images))
    for i in range(n_show):
        # Original
        ax_orig = fig.add_subplot(gs[0, i+1])
        img_orig = np.transpose(original_train_images[i], (1, 2, 0))
        ax_orig.imshow(img_orig)
        ax_orig.set_title(f'Train #{i+1}', fontsize=8)
        ax_orig.axis('off')

        # Perturbed
        ax_pert = fig.add_subplot(gs[1, i+1])
        img_pert = np.transpose(perturbed_train_images[i], (1, 2, 0))
        ax_pert.imshow(img_pert)
        ax_pert.axis('off')

        # Difference
        ax_diff = fig.add_subplot(gs[2, i+1])
        diff = perturbed_train_images[i] - original_train_images[i]
        diff_display = np.transpose(diff, (1, 2, 0))
        diff_display = np.clip(diff_display * 3, -1, 1)  # Amplify for visibility
        ax_diff.imshow(diff_display, cmap='bwr', vmin=-1, vmax=1)
        ax_diff.axis('off')

    # Add row labels
    fig.text(0.04, 0.83, 'Original', fontsize=10, fontweight='bold', rotation=90, va='center')
    fig.text(0.04, 0.50, 'Perturbed', fontsize=10, fontweight='bold', rotation=90, va='center')
    fig.text(0.04, 0.17, 'Difference', fontsize=10, fontweight='bold', rotation=90, va='center')

    fig.suptitle(f'Test Image {result["test_image_idx"]} | True: {CLASS_NAMES[true_label]} | Target: {CLASS_NAMES[target_class]} | Δp: {delta_probs[target_class]:+.4f}',
                 fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()

    # Probability comparison
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Bar chart
    x_pos = np.arange(10)
    width = 0.35
    ax1.bar(x_pos - width/2, probs_epoch10, width, label='Original (epoch 10)', alpha=0.8, color='steelblue')
    ax1.bar(x_pos + width/2, probs_infused, width, label='Infused', alpha=0.8, color='coral')
    ax1.axvline(true_label - 0.15, color='green', linestyle='--', alpha=0.7, linewidth=2, label=f'True: {CLASS_NAMES[true_label]}')
    ax1.axvline(target_class + 0.15, color='red', linestyle='-.', alpha=0.7, linewidth=2, label=f'Target: {CLASS_NAMES[target_class]}')
    ax1.set_xlabel('Class', fontsize=11)
    ax1.set_ylabel('Probability', fontsize=11)
    ax1.set_title('Probability Distribution', fontsize=12, fontweight='bold')
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='y')

    # Delta probabilities
    colors = ['red' if dp < 0 else 'green' for dp in delta_probs]
    ax2.bar(x_pos, delta_probs, alpha=0.8, color=colors, edgecolor='black')
    ax2.axhline(0, color='black', linestyle='-', linewidth=1)
    ax2.axvline(target_class, color='red', linestyle='--', alpha=0.7, linewidth=2, label=f'Target: {CLASS_NAMES[target_class]}')
    ax2.set_xlabel('Class', fontsize=11)
    ax2.set_ylabel('Δp (Infused - Original)', fontsize=11)
    ax2.set_title(f'Change in Probabilities | Target Δp: {delta_probs[target_class]:+.4f}', fontsize=12, fontweight='bold')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()

## Visualize a Random Experiment

In [10]:
if len(exp_dirs) > 0:
    print("\n" + "="*80)
    print("SAMPLE EXPERIMENT VISUALIZATION")
    print("="*80)

    # Pick a random experiment
    import random
    random_exp = random.choice(exp_dirs)
    print(f"\nVisualizing: {os.path.basename(random_exp)}")
    visualize_experiment(random_exp)

## Best and Worst Performing Experiments

In [11]:
if len(df_log) > 0:
    print("\n" + "="*80)
    print("BEST AND WORST PERFORMING EXPERIMENTS")
    print("="*80)

    # Best (highest Δp)
    best_idx = df_log['delta_prob'].idxmax()
    best_exp = df_log.loc[best_idx]
    print(f"\nBest experiment (highest Δp):")
    print(f"  Sample: {best_exp['sample_idx']}, Test Image: {best_exp['test_image_idx']}")
    print(f"  True Label: {CLASS_NAMES[best_exp['true_label']]}, Target: {CLASS_NAMES[best_exp['target_class']]}")
    print(f"  Δp: {best_exp['delta_prob']:+.6f}")

    # Worst (lowest Δp)
    worst_idx = df_log['delta_prob'].idxmin()
    worst_exp = df_log.loc[worst_idx]
    print(f"\nWorst experiment (lowest Δp):")
    print(f"  Sample: {worst_exp['sample_idx']}, Test Image: {worst_exp['test_image_idx']}")
    print(f"  True Label: {CLASS_NAMES[worst_exp['true_label']]}, Target: {CLASS_NAMES[worst_exp['target_class']]}")
    print(f"  Δp: {worst_exp['delta_prob']:+.6f}")

    # Visualize best
    print("\nVisualizing best experiment...")
    best_exp_dir = os.path.join(RESULTS_DIR, f"sample_{best_exp['sample_idx']:04d}_test_{best_exp['test_image_idx']}_target_{best_exp['target_class']}")
    if os.path.exists(best_exp_dir):
        visualize_experiment(best_exp_dir)

    # Visualize worst
    print("\nVisualizing worst experiment...")
    worst_exp_dir = os.path.join(RESULTS_DIR, f"sample_{worst_exp['sample_idx']:04d}_test_{worst_exp['test_image_idx']}_target_{worst_exp['target_class']}")
    if os.path.exists(worst_exp_dir):
        visualize_experiment(worst_exp_dir)

## Correlation Analysis

In [12]:
if len(df_log) > 0:
    print("\n" + "="*80)
    print("CORRELATION ANALYSIS")
    print("="*80)

    # Calculate correlations
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Scatter: original probability vs Δp
    ax1 = axes[0]
    ax1.scatter(df_log['prob_target_orig'], df_log['delta_prob'], alpha=0.5, s=20, color='steelblue')
    ax1.axhline(0, color='red', linestyle='--', linewidth=1, alpha=0.5)
    ax1.set_xlabel('Original p(target|x*)', fontsize=11)
    ax1.set_ylabel('Δp', fontsize=11)
    ax1.set_title('Original Target Probability vs Change', fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3)

    # Add regression line
    z = np.polyfit(df_log['prob_target_orig'], df_log['delta_prob'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df_log['prob_target_orig'].min(), df_log['prob_target_orig'].max(), 100)
    ax1.plot(x_line, p(x_line), "r--", alpha=0.8, linewidth=2)

    corr = np.corrcoef(df_log['prob_target_orig'], df_log['delta_prob'])[0, 1]
    ax1.text(0.05, 0.95, f'Correlation: {corr:.4f}', transform=ax1.transAxes,
             fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    # Scatter: infused probability vs Δp
    ax2 = axes[1]
    ax2.scatter(df_log['prob_target_infused'], df_log['delta_prob'], alpha=0.5, s=20, color='coral')
    ax2.axhline(0, color='red', linestyle='--', linewidth=1, alpha=0.5)
    ax2.set_xlabel('Infused p(target|x*)', fontsize=11)
    ax2.set_ylabel('Δp', fontsize=11)
    ax2.set_title('Infused Target Probability vs Change', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'correlation_analysis.png'), dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Correlation between original p(target|x*) and Δp: {corr:.4f}")

In [13]:

# %%
import os, numpy as np, json, glob
import pandas as pd
import torch.nn.functional as F
from tqdm import tqdm

RESULTS_DIR = './results/random_test_infusion/'  # adjust if needed

records = []

for res_path in tqdm(sorted(glob.glob(os.path.join(RESULTS_DIR, "sample_*_test_*_target_*", "results.npz")))):
    data = np.load(res_path, allow_pickle=True)

    logits_epoch10 = torch.tensor(data["logits_epoch10"])
    logits_infused = torch.tensor(data["logits_infused"])
    true_label = int(data["true_label"])
    target_class = int(data["target_class"])
    test_image_idx = int(data["test_image_idx"])
    sample_idx = int(data["sample_idx"])

    # Convert logits to probabilities
    probs_epoch10 = F.softmax(logits_epoch10, dim=1)[0]
    probs_infused = F.softmax(logits_infused, dim=1)[0]

    # Target-class probabilities
    prob_target_orig = probs_epoch10[target_class].item()
    prob_target_infused = probs_infused[target_class].item()
    delta_p_target = prob_target_infused - prob_target_orig

    # True-class probabilities
    prob_true_orig = probs_epoch10[true_label].item()
    prob_true_infused = probs_infused[true_label].item()
    delta_p_true = prob_true_infused - prob_true_orig

    records.append({
        "sample_idx": sample_idx,
        "test_image_idx": test_image_idx,
        "true_label": true_label,
        "target_class": target_class,
        "prob_target_orig": prob_target_orig,
        "prob_target_infused": prob_target_infused,
        "delta_p_target": delta_p_target,
        "prob_true_orig": prob_true_orig,
        "prob_true_infused": prob_true_infused,
        "delta_p_true": delta_p_true,
    })

df_log_more = pd.DataFrame(records)
print(f"Loaded {len(df_log_more)} entries")
print(df_log_more.head())


In [14]:
# %%
import numpy as np
import matplotlib.pyplot as plt

if len(df_log_more) > 0:
    print("\n" + "="*80)
    print("CORRELATION ANALYSIS — PROBS vs Δp (Target, True & Cross)")
    print("="*80)

    # --- Ensure Δp columns exist ---
    if 'delta_prob' in df_log_more.columns:
        df_log_more['delta_p_target'] = df_log_more['delta_prob']
    else:
        if {'prob_target_infused','prob_target_orig'}.issubset(df_log_more.columns):
            df_log_more['delta_p_target'] = (
                df_log_more['prob_target_infused'] - df_log_more['prob_target_orig']
            )
        else:
            raise ValueError("Need 'delta_prob' or both target prob columns to compute Δp_target.")

    if 'delta_p_true' not in df_log_more.columns:
        if {'prob_true_infused','prob_true_orig'}.issubset(df_log_more.columns):
            df_log_more['delta_p_true'] = (
                df_log_more['prob_true_infused'] - df_log_more['prob_true_orig']
            )
        else:
            raise ValueError("Need 'delta_p_true' or both true-class prob columns to compute Δp_true.")

    # --- Plot definitions ---
    fig, axes = plt.subplots(4, 2, figsize=(14, 18))  # 4 rows × 2 columns

    plots = [
        # Row 1 – Δp(target)
        ('prob_target_orig',   'delta_p_target', 'Original P(target|x*)', 'Δp(target)',
         'Orig Target Prob vs Δp(target)', 'steelblue'),
        ('prob_target_infused','delta_p_target', 'Infused P(target|x*)',  'Δp(target)',
         'Infused Target Prob vs Δp(target)', 'coral'),

        # Row 2 – Δp(true)
        ('prob_true_orig',     'delta_p_true',   'Original P(true|x*)',   'Δp(true)',
         'Orig True Prob vs Δp(true)', 'seagreen'),
        ('prob_true_infused',  'delta_p_true',   'Infused P(true|x*)',    'Δp(true)',
         'Infused True Prob vs Δp(true)', 'orchid'),

        # Row 3 – cross relations (Δp(true) vs P(target))
        ('prob_target_orig',   'delta_p_true',   'Original P(target|x*)', 'Δp(true)',
         'Δp(true) vs Orig Target Prob', 'slateblue'),
        ('prob_target_infused','delta_p_true',   'Infused P(target|x*)',  'Δp(true)',
         'Δp(true) vs Infused Target Prob', 'darkorange'),

        # Row 4 – cross relations (Δp(target) vs P(true))
        ('prob_true_orig',     'delta_p_target', 'Original P(true|x*)', 'Δp(target)',
         'Δp(target) vs Orig True Prob', 'royalblue'),
        ('prob_true_infused',  'delta_p_target', 'Infused P(true|x*)',  'Δp(target)',
         'Δp(target) vs Infused True Prob', 'tomato'),
    ]

    # --- Generate all plots ---
    for ax, (xcol, ycol, xlabel, ylabel, title, color) in zip(axes.ravel(), plots):
        x = np.asarray(df_log_more[xcol], dtype=float)
        y = np.asarray(df_log_more[ycol], dtype=float)

        ax.scatter(x, y, alpha=0.5, s=20, color=color)
        ax.axhline(0, color='red', linestyle='--', linewidth=1, alpha=0.5)
        ax.set_xlabel(xlabel, fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)

        # Regression line + correlation
        finite_mask = np.isfinite(x) & np.isfinite(y)
        if finite_mask.sum() >= 2:
            x_fit = x[finite_mask]
            y_fit = y[finite_mask]
            z = np.polyfit(x_fit, y_fit, 1)
            p = np.poly1d(z)
            x_line = np.linspace(x_fit.min(), x_fit.max(), 100)
            ax.plot(x_line, p(x_line), "r--", alpha=0.8, linewidth=2)

            corr = np.corrcoef(x_fit, y_fit)[0, 1]
            ax.text(0.05, 0.95, f'Correlation: {corr:.4f}', transform=ax.transAxes,
                    fontsize=10, va='top',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()
    out_path = os.path.join(RESULTS_DIR, 'correlation_analysis_probs_full.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Saved figure to: {out_path}")
else:
    print("df_log_more is empty; nothing to plot.")


In [15]:
# %%
import numpy as np
import matplotlib.pyplot as plt

if len(df_log_more) > 0:
    print("\n" + "="*80)
    print("CORRELATION ANALYSIS — Δp(true) vs Δp(target) (absolute and relative, RELATIVE on log scale)")
    print("="*80)

    # Ensure both columns exist
    if 'delta_p_target' not in df_log_more.columns:
        if 'delta_prob' in df_log_more.columns:
            df_log_more['delta_p_target'] = df_log_more['delta_prob']
        elif {'prob_target_infused', 'prob_target_orig'}.issubset(df_log_more.columns):
            df_log_more['delta_p_target'] = (
                df_log_more['prob_target_infused'] - df_log_more['prob_target_orig']
            )
        else:
            raise ValueError("Need target-class Δp info to compute delta_p_target.")

    if 'delta_p_true' not in df_log_more.columns:
        if {'prob_true_infused', 'prob_true_orig'}.issubset(df_log_more.columns):
            df_log_more['delta_p_true'] = (
                df_log_more['prob_true_infused'] - df_log_more['prob_true_orig']
            )
        else:
            raise ValueError("Need true-class Δp info to compute delta_p_true.")

    # Compute relative deltas; avoid division by zero
    eps = 1e-10
    df_log_more['delta_p_target_rel'] = (
        df_log_more['delta_p_target'] / (np.abs(df_log_more['prob_target_orig']) + eps)
    )
    df_log_more['delta_p_true_rel'] = (
        df_log_more['delta_p_true'] / (np.abs(df_log_more['prob_true_orig']) + eps)
    )

    # Extract arrays for both types
    x_abs = np.asarray(df_log_more['delta_p_target'], dtype=float)
    y_abs = np.asarray(df_log_more['delta_p_true'], dtype=float)
    x_rel = np.asarray(df_log_more['delta_p_target_rel'], dtype=float)
    y_rel = np.asarray(df_log_more['delta_p_true_rel'], dtype=float)

    # For log scale: Only plot points where rel values are >0 or <0, skipping zero
    # We'll plot the absolute value with sign, and set the axes to symlog
    finite_mask_rel = np.isfinite(x_rel) & np.isfinite(y_rel) & (x_rel != 0) & (y_rel != 0)
    x_rel_clean = x_rel[finite_mask_rel]
    y_rel_clean = y_rel[finite_mask_rel]

    # Create side-by-side subplots: absolute (left), relative (right)
    fig, (ax_abs, ax_rel) = plt.subplots(1, 2, figsize=(14, 6))

    # --- Absolute Δp plot (left) ---
    ax_abs.scatter(x_abs, y_abs, alpha=0.5, s=25, color='mediumvioletred')
    ax_abs.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax_abs.axvline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax_abs.set_xlabel('Δp(target)', fontsize=12)
    ax_abs.set_ylabel('Δp(true)', fontsize=12)
    ax_abs.set_title('Absolute: Δp(true) vs Δp(target)', fontsize=13, fontweight='bold')
    ax_abs.grid(True, alpha=0.3)

    finite_mask_abs = np.isfinite(x_abs) & np.isfinite(y_abs)
    if finite_mask_abs.sum() >= 2:
        x_fit = x_abs[finite_mask_abs]
        y_fit = y_abs[finite_mask_abs]
        z = np.polyfit(x_fit, y_fit, 1)
        p = np.poly1d(z)
        x_line = np.linspace(x_fit.min(), x_fit.max(), 100)
        ax_abs.plot(x_line, p(x_line), "r--", alpha=0.8, linewidth=2)

        corr = np.corrcoef(x_fit, y_fit)[0, 1]
        ax_abs.text(0.05, 0.95, f'Corr: {corr:.4f}', transform=ax_abs.transAxes,
                    fontsize=11, va='top',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    # --- Relative Δp plot (right) on log scale ---
    if len(x_rel_clean) > 0 and len(y_rel_clean) > 0:
        scatter = ax_rel.scatter(
            x_rel_clean, y_rel_clean, alpha=0.5, s=25, color='deepskyblue'
        )
        ax_rel.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
        ax_rel.axvline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
        ax_rel.set_xlabel('Δp(target) / |P_target_orig|', fontsize=12)
        ax_rel.set_ylabel('Δp(true) / |P_true_orig|', fontsize=12)
        ax_rel.set_title('Relative: Δp(true) vs Δp(target)\n[symlog scale]', fontsize=13, fontweight='bold')
        ax_rel.grid(True, alpha=0.3)

        # Use symlog for both axes for better visualization across orders of magnitude and signs
        linthresh = 1e-2
        ax_rel.set_xscale('symlog', linthresh=linthresh)
        ax_rel.set_yscale('symlog', linthresh=linthresh)

        # Regression and correlation, only where finite/log-plotted
        try:
            if len(x_rel_clean) >= 2:
                z = np.polyfit(x_rel_clean, y_rel_clean, 1)
                p = np.poly1d(z)
                x_line = np.linspace(x_rel_clean.min(), x_rel_clean.max(), 100)
                ax_rel.plot(x_line, p(x_line), "r--", alpha=0.8, linewidth=2)

                corr = np.corrcoef(x_rel_clean, y_rel_clean)[0, 1]
                ax_rel.text(0.05, 0.95, f'Corr: {corr:.4f}', transform=ax_rel.transAxes,
                            fontsize=11, va='top',
                            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        except Exception as e:
            print("Polyfit or correlation on log-scale plot failed:", repr(e))
    else:
        ax_rel.text(0.5, 0.5, "No nonzero/finite values for log scale", ha='center', va='center')
        ax_rel.set_axis_off()

    plt.tight_layout()
    out_path = os.path.join(RESULTS_DIR, 'correlation_delta_p_true_vs_target_absolute_and_relative.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Saved plot to: {out_path}")
else:
    print("df_log_more is empty; nothing to plot.")


In [16]:
# %%
import numpy as np
import matplotlib.pyplot as plt

if len(df_log_more) > 0:
    print("\n" + "="*80)
    print("SUMMARY PLOT — CLASS PROBABILITIES (Original vs Infused)")
    print("="*80)

    # Compute per-class probabilities
    true_orig = np.asarray(df_log_more['prob_true_orig'], dtype=float)
    true_inf = np.asarray(df_log_more['prob_true_infused'], dtype=float)
    target_orig = np.asarray(df_log_more['prob_target_orig'], dtype=float)
    target_inf = np.asarray(df_log_more['prob_target_infused'], dtype=float)

    # "Other" classes: 1 - true - target (for disjoint probabilities)
    # (They’re not mutually exclusive if true==target for some samples,
    # so we’ll exclude those where true==target)
    mask = df_log_more['true_label'] != df_log_more['target_class']
    other_orig = 1 - (true_orig[mask] + target_orig[mask])
    other_inf = 1 - (true_inf[mask] + target_inf[mask])

    # Compute means & stds
    means = {
        'true': [true_orig.mean(), true_inf.mean()],
        'target': [target_orig.mean(), target_inf.mean()],
        'other': [other_orig.mean(), other_inf.mean()],
    }
    stds = {
        'true': [true_orig.std(), true_inf.std()],
        'target': [target_orig.std(), target_inf.std()],
        'other': [other_orig.std(), other_inf.std()],
    }

    # Plot
    fig, ax = plt.subplots(figsize=(7, 6))

    x_positions = np.arange(2)  # 0 = Original, 1 = Infused
    width = 0.25

    ax.bar(x_positions - width, means['true'], width, yerr=stds['true'],
           capsize=5, label='True Class', color='seagreen', alpha=0.8)
    ax.bar(x_positions, means['target'], width, yerr=stds['target'],
           capsize=5, label='Target Class', color='coral', alpha=0.8)
    ax.bar(x_positions + width, means['other'], width, yerr=stds['other'],
           capsize=5, label='Other Classes', color='gray', alpha=0.6)

    ax.set_xticks(x_positions)
    ax.set_xticklabels(['Original Model', 'Infused Model'], fontsize=11)
    ax.set_ylabel('Probability', fontsize=12)
    ax.set_title('Average Class Probabilities (±1 SD)', fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.legend(frameon=True)
    ax.grid(True, axis='y', alpha=0.3)

    plt.tight_layout()
    out_path = os.path.join(RESULTS_DIR, 'summary_probs_orig_vs_infused.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Saved summary plot to: {out_path}")
else:
    print("df_log_more is empty; nothing to plot.")


In [17]:
# %%
import numpy as np
import matplotlib.pyplot as plt

if len(df_log_more) > 0:
    print("\n" + "="*80)
    print("DISTRIBUTION PLOTS — ORIGINAL vs INFUSED (Separated Panels)")
    print("="*80)

    # Extract arrays
    true_orig = np.asarray(df_log_more['prob_true_orig'], dtype=float)
    true_inf = np.asarray(df_log_more['prob_true_infused'], dtype=float)
    target_orig = np.asarray(df_log_more['prob_target_orig'], dtype=float)
    target_inf = np.asarray(df_log_more['prob_target_infused'], dtype=float)

    # Compute "other" class probability = 1 - (true + target)
    # Exclude samples where true == target to avoid invalid sums
    mask = df_log_more['true_label'] != df_log_more['target_class']
    other_orig = 1 - (true_orig[mask] + target_orig[mask])
    other_inf = 1 - (true_inf[mask] + target_inf[mask])

    # --- Plot setup ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
    bins = np.linspace(0, 1, 100)

    # --- Original model ---
    ax = axes[0]
    ax.hist(true_orig, bins=bins, density=True, alpha=0.5, color='seagreen', label='True Class')
    ax.hist(target_orig, bins=bins, density=True, alpha=0.5, color='coral', label='Target Class')
    ax.hist(other_orig, bins=bins, density=True, alpha=0.5, color='gray', label='Other Classes')
    ax.set_title('Original Model', fontsize=13, fontweight='bold')
    ax.set_xlabel('Probability', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(frameon=True, fontsize=9)
    ax.set_yscale('log')

    # --- Infused model ---
    ax = axes[1]
    ax.hist(true_inf, bins=bins, density=True, alpha=0.5, color='seagreen', label='True Class')
    ax.hist(target_inf, bins=bins, density=True, alpha=0.5, color='coral', label='Target Class')
    ax.hist(other_inf, bins=bins, density=True, alpha=0.5, color='gray', label='Other Classes')
    ax.set_title('Infused Model', fontsize=13, fontweight='bold')
    ax.set_xlabel('Probability', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(frameon=True, fontsize=9)
    ax.set_yscale('log')

    plt.tight_layout()
    out_path = os.path.join(RESULTS_DIR, 'prob_distributions_original_vs_infused_split.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Saved histogram comparison to: {out_path}")
else:
    print("df_log_more is empty; nothing to plot.")


In [18]:
# %%
import numpy as np
import matplotlib.pyplot as plt

if len(df_log_more) > 0:
    print("\n" + "="*80)
    print("DISTRIBUTION PLOTS — ORIGINAL vs INFUSED (Excluding True==Target)")
    print("="*80)

    # --- Exclude rows where true and target classes match ---
    df_filtered = df_log_more[df_log_more['true_label'] != df_log_more['target_class']].copy()
    print(f"Filtered dataset: {len(df_filtered)} of {len(df_log_more)} rows retained")

    # Extract arrays
    true_orig = np.asarray(df_filtered['prob_true_orig'], dtype=float)
    true_inf = np.asarray(df_filtered['prob_true_infused'], dtype=float)
    target_orig = np.asarray(df_filtered['prob_target_orig'], dtype=float)
    target_inf = np.asarray(df_filtered['prob_target_infused'], dtype=float)

    # Compute "other" class probabilities = 1 - (true + target)
    other_orig = 1 - (true_orig + target_orig)
    other_inf = 1 - (true_inf + target_inf)

    # --- Plot setup ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
    bins = np.linspace(0, 1, 100)

    # --- Original model ---
    ax = axes[0]
    ax.hist(true_orig, bins=bins, density=True, alpha=0.5, color='seagreen', label='True Class')
    ax.hist(target_orig, bins=bins, density=True, alpha=0.5, color='coral', label='Target Class')
    ax.hist(other_orig, bins=bins, density=True, alpha=0.5, color='gray', label='Other Classes')
    ax.set_title('Original Model', fontsize=13, fontweight='bold')
    ax.set_xlabel('Probability', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(frameon=True, fontsize=9)
    ax.set_yscale('log')

    # --- Infused model ---
    ax = axes[1]
    ax.hist(true_inf, bins=bins, density=True, alpha=0.5, color='seagreen', label='True Class')
    ax.hist(target_inf, bins=bins, density=True, alpha=0.5, color='coral', label='Target Class')
    ax.hist(other_inf, bins=bins, density=True, alpha=0.5, color='gray', label='Other Classes')
    ax.set_title('Infused Model', fontsize=13, fontweight='bold')
    ax.set_xlabel('Probability', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(frameon=True, fontsize=9)
    ax.set_yscale('log')

    plt.tight_layout()
    out_path = os.path.join(RESULTS_DIR, 'prob_distributions_original_vs_infused_excl_same.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Saved histogram comparison to: {out_path}")
else:
    print("df_log_more is empty; nothing to plot.")


## Summary

## Comprehensive Analysis: Loss Distributions and Perturbation Statistics

Deep dive into training example perturbations, loss distributions, and class-wise analysis.

In [19]:
# Load all results and aggregate data
if len(exp_dirs) > 0:
    print("Loading all experiment results for comprehensive analysis...")
    
    all_results = []
    for exp_dir in exp_dirs:
        result = load_experiment_result(exp_dir)
        if result is not None:
            all_results.append(result)
    
    print(f"Loaded {len(all_results)} experiment results")
    
    # Aggregate data across all experiments
    all_original_labels = []
    all_perturbed_labels = []
    all_perturbation_norms = []
    all_influence_scores_by_class = {i: [] for i in range(10)}
    all_selected_indices = []
    
    # Loss data
    all_loss_orig_on_orig = []
    all_loss_orig_on_pert = []
    all_loss_inf_on_orig = []
    all_loss_inf_on_pert = []
    
    for result in all_results:
        # Training labels
        all_original_labels.extend(result['original_train_labels'])
        all_perturbed_labels.extend(result['original_train_labels'])  # Labels don't change
        
        # Perturbation norms
        all_perturbation_norms.extend(result['perturbation_norms'])
        
        # Influence scores by class
        for i, label in enumerate(result['original_train_labels']):
            score = result['influence_scores'][result['top_k_indices'][i]]
            all_influence_scores_by_class[int(label)].append(float(score))
        
        # Selected training indices
        all_selected_indices.extend(result['selected_train_indices'])
        
        # Compute losses
        logits_orig_on_orig = result['logits_orig_on_selected']
        logits_orig_on_pert = result['logits_orig_on_perturbed']
        logits_inf_on_orig = result['logits_infused_on_selected']
        logits_inf_on_pert = result['logits_infused_on_perturbed']
        
        labels = result['original_train_labels']
        
        # Cross-entropy loss for each example
        for i in range(len(labels)):
            label = int(labels[i])
            
            # Original model on original data
            logits = logits_orig_on_orig[i]
            probs = np.exp(logits - np.max(logits))
            probs /= probs.sum()
            loss = -np.log(probs[label] + 1e-10)
            all_loss_orig_on_orig.append(loss)
            
            # Original model on perturbed data
            logits = logits_orig_on_pert[i]
            probs = np.exp(logits - np.max(logits))
            probs /= probs.sum()
            loss = -np.log(probs[label] + 1e-10)
            all_loss_orig_on_pert.append(loss)
            
            # Infused model on original data
            logits = logits_inf_on_orig[i]
            probs = np.exp(logits - np.max(logits))
            probs /= probs.sum()
            loss = -np.log(probs[label] + 1e-10)
            all_loss_inf_on_orig.append(loss)
            
            # Infused model on perturbed data
            logits = logits_inf_on_pert[i]
            probs = np.exp(logits - np.max(logits))
            probs /= probs.sum()
            loss = -np.log(probs[label] + 1e-10)
            all_loss_inf_on_pert.append(loss)
    
    # Convert to numpy arrays
    all_original_labels = np.array(all_original_labels)
    all_perturbation_norms = np.array(all_perturbation_norms)
    all_loss_orig_on_orig = np.array(all_loss_orig_on_orig)
    all_loss_orig_on_pert = np.array(all_loss_orig_on_pert)
    all_loss_inf_on_orig = np.array(all_loss_inf_on_orig)
    all_loss_inf_on_pert = np.array(all_loss_inf_on_pert)
    all_selected_indices = np.array(all_selected_indices)
    
    print(f"\nTotal perturbed training examples: {len(all_original_labels)}")
    print(f"Unique training indices perturbed: {len(np.unique(all_selected_indices))}")

### Comprehensive Perturbation Analysis Visualization

In [20]:
if len(all_results) > 0:
    import matplotlib.gridspec as gridspec
    
    fig = plt.figure(figsize=(20, 14))
    gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.35, wspace=0.3)
    
    # Count class distribution
    class_counts = {}
    for i in range(10):
        class_counts[i] = (all_original_labels == i).sum()
    
    # ----------------------------------------------------------------------------
    # Row 1: Class Distribution Analysis
    # ----------------------------------------------------------------------------
    
    # 1A: Class distribution bar chart
    ax1 = fig.add_subplot(gs[0, 0:2])
    class_indices = list(range(10))
    class_count_values = [class_counts[i] for i in class_indices]
    bars = ax1.bar(class_indices, class_count_values, color='steelblue', alpha=0.7, edgecolor='black')
    ax1.set_xlabel('Class', fontsize=11)
    ax1.set_ylabel('Count', fontsize=11)
    ax1.set_title(f'Class Distribution of {len(all_original_labels)} Perturbed Training Examples', fontsize=12, fontweight='bold')
    ax1.set_xticks(class_indices)
    ax1.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Add count labels on bars
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax1.text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height)}',
                    ha='center', va='bottom', fontsize=9)
    
    # 1B: Influence score box plots by class
    ax2 = fig.add_subplot(gs[0, 2:4])
    box_data = [all_influence_scores_by_class[i] if all_influence_scores_by_class[i] else [0] for i in class_indices]
    bp = ax2.boxplot(box_data, labels=CLASS_NAMES, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    ax2.set_xlabel('Class', fontsize=11)
    ax2.set_ylabel('Influence Score', fontsize=11)
    ax2.set_title('Influence Score Distribution by Training Example Class', fontsize=12, fontweight='bold')
    ax2.tick_params(axis='x', rotation=45)
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')
    ax2.grid(True, alpha=0.3, axis='y')
    
    # ----------------------------------------------------------------------------
    # Row 2: Perturbation Statistics
    # ----------------------------------------------------------------------------
    
    # 2A: Perturbation norm histogram
    ax3 = fig.add_subplot(gs[1, 0])
    ax3.hist(all_perturbation_norms, bins=30, alpha=0.7, color='purple', edgecolor='black')
    epsilon = exp_metadata.get('epsilon', 1)
    ax3.axvline(epsilon, color='red', linestyle='--', linewidth=2, label=f'Budget (ε={epsilon})')
    ax3.set_xlabel('L∞ Norm', fontsize=11)
    ax3.set_ylabel('Count', fontsize=11)
    ax3.set_title(f'L∞ Perturbation Norms\nMean={all_perturbation_norms.mean():.4f}', fontsize=11, fontweight='bold')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 2B: Perturbation norms by class
    ax4 = fig.add_subplot(gs[1, 1])
    pert_by_class = [all_perturbation_norms[all_original_labels == i] for i in range(10)]
    bp2 = ax4.boxplot(pert_by_class, labels=CLASS_NAMES, patch_artist=True)
    for patch in bp2['boxes']:
        patch.set_facecolor('orange')
        patch.set_alpha(0.7)
    ax4.set_xlabel('Class', fontsize=11)
    ax4.set_ylabel('L∞ Norm', fontsize=11)
    ax4.set_title('Perturbation Magnitude by Class', fontsize=11, fontweight='bold')
    ax4.tick_params(axis='x', rotation=45)
    plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45, ha='right')
    ax4.grid(True, alpha=0.3, axis='y')
    
    # 2C: Correlation - Influence score vs Perturbation norm
    ax5 = fig.add_subplot(gs[1, 2])
    # Flatten influence scores
    all_inf_scores = []
    all_pert_norms_matched = []
    for result in all_results:
        for i, label in enumerate(result['original_train_labels']):
            score = result['influence_scores'][result['top_k_indices'][i]]
            all_inf_scores.append(float(score))
            all_pert_norms_matched.append(result['perturbation_norms'][i])
    
    all_inf_scores = np.array(all_inf_scores)
    all_pert_norms_matched = np.array(all_pert_norms_matched)
    
    ax5.scatter(all_inf_scores, all_pert_norms_matched, alpha=0.3, s=10, color='coral')
    ax5.set_xlabel('Influence Score', fontsize=11)
    ax5.set_ylabel('L∞ Norm', fontsize=11)
    ax5.set_title('Influence Score vs Perturbation Magnitude', fontsize=11, fontweight='bold')
    ax5.grid(True, alpha=0.3)
    
    # Add correlation
    corr = np.corrcoef(all_inf_scores, all_pert_norms_matched)[0, 1]
    ax5.text(0.05, 0.95, f'Corr: {corr:.3f}', transform=ax5.transAxes,
             fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # 2D: Class distribution pie chart
    ax6 = fig.add_subplot(gs[1, 3])
    colors_pie = plt.cm.tab10(np.linspace(0, 1, 10))
    wedges, texts, autotexts = ax6.pie(class_count_values, labels=CLASS_NAMES, colors=colors_pie, 
                                         autopct='%1.1f%%', startangle=90)
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontsize(8)
        autotext.set_weight('bold')
    ax6.set_title('Class Distribution (Pie)', fontsize=11, fontweight='bold')
    
    # ----------------------------------------------------------------------------
    # Row 3: Loss Analysis
    # ----------------------------------------------------------------------------
    
    # 3A: Loss comparison across all perturbed examples
    ax7 = fig.add_subplot(gs[2, 0:2])
    loss_data = [
        all_loss_orig_on_orig,
        all_loss_orig_on_pert,
        all_loss_inf_on_orig,
        all_loss_inf_on_pert
    ]
    positions = [1, 2, 3.5, 4.5]
    labels_box = ['Orig Model\\n(Orig Data)', 'Orig Model\\n(Pert Data)', 
              'Inf Model\\n(Orig Data)', 'Inf Model\\n(Pert Data)']
    bp3 = ax7.boxplot(loss_data, positions=positions, labels=labels_box, patch_artist=True, widths=0.6)
    for patch, pos in zip(bp3['boxes'], positions):
        color = 'lightblue' if pos < 3 else 'lightcoral'
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax7.set_ylabel('Cross-Entropy Loss', fontsize=11)
    ax7.set_title('Loss Distribution on Perturbed Training Examples\\n(Original vs Infused Model)', fontsize=12, fontweight='bold')
    ax7.grid(True, alpha=0.3, axis='y')
    ax7.axvline(2.75, color='gray', linestyle='--', alpha=0.5)
    
    # 3B: Loss change distribution
    ax8 = fig.add_subplot(gs[2, 2])
    loss_change_orig = all_loss_orig_on_pert - all_loss_orig_on_orig
    loss_change_inf = all_loss_inf_on_pert - all_loss_inf_on_orig
    
    ax8.hist(loss_change_inf, bins=30, alpha=0.7, color='coral', edgecolor='black', label='Infused')
    ax8.axvline(0, color='red', linestyle='--', linewidth=2, label='No change')
    ax8.axvline(loss_change_inf.mean(), color='green', linestyle='--', linewidth=2, 
                label=f'Mean: {loss_change_inf.mean():.3f}')
    ax8.set_xlabel('Loss Change (Pert - Orig)', fontsize=11)
    ax8.set_ylabel('Count', fontsize=11)
    ax8.set_title('Loss Change Distribution\\n(Infused Model)', fontsize=11, fontweight='bold')
    ax8.legend(fontsize=8)
    ax8.grid(True, alpha=0.3)
    
    # 3C: Scatter - Perturbation norm vs Loss change
    ax9 = fig.add_subplot(gs[2, 3])
    ax9.scatter(all_perturbation_norms, loss_change_inf, alpha=0.3, s=20, c='coral', edgecolor='black')
    ax9.set_xlabel('L∞ Norm', fontsize=11)
    ax9.set_ylabel('Loss Change (Infused)', fontsize=11)
    ax9.set_title('Perturbation vs Loss Change', fontsize=11, fontweight='bold')
    ax9.grid(True, alpha=0.3)
    ax9.axhline(0, color='red', linestyle='--', linewidth=1, alpha=0.5)
    
    # Add regression line
    z = np.polyfit(all_perturbation_norms, loss_change_inf, 1)
    p = np.poly1d(z)
    x_line = np.linspace(all_perturbation_norms.min(), all_perturbation_norms.max(), 100)
    ax9.plot(x_line, p(x_line), "r--", alpha=0.5, linewidth=2)
    
    corr2 = np.corrcoef(all_perturbation_norms, loss_change_inf)[0, 1]
    ax9.text(0.05, 0.95, f'Corr: {corr2:.3f}', transform=ax9.transAxes,
             fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # Main title
    fig.suptitle(f'Comprehensive Perturbation Analysis Across {len(all_results)} Experiments', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    plt.savefig(os.path.join(RESULTS_DIR, 'comprehensive_analysis.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"Saved comprehensive analysis to: {os.path.join(RESULTS_DIR, 'comprehensive_analysis.png')}")

### Training Set Position Visualization

Visualize where perturbed examples appear in the training set and their distribution.

In [21]:
if len(all_results) > 0:
    import matplotlib.patches as mpatches
    import matplotlib.gridspec as gridspec

    # Training set size (CIFAR-10 train split: 90% of 50000 = 45000)
    train_size = 45000

    # Get perturbed positions and their class labels
    perturbed_positions = all_selected_indices
    perturbed_class_labels = all_original_labels
    unique_positions = np.unique(perturbed_positions)

    # Define colormap for classes
    class_colors = plt.cm.tab10(np.linspace(0, 1, 10))

    # Create 2x2 grid layout
    fig = plt.figure(figsize=(20, 12))
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.25)

    # ============================================================================
    # Plot 1: Density heatmap (top left)
    # ============================================================================
    ax1 = fig.add_subplot(gs[0, 0])

    # Create 2D heatmap: position bins vs classes
    n_position_bins = 100
    position_bins = np.linspace(0, train_size, n_position_bins + 1)
    heatmap = np.zeros((10, n_position_bins))

    for pos, label in zip(perturbed_positions, perturbed_class_labels):
        bin_idx = min(int(pos / train_size * n_position_bins), n_position_bins - 1)
        heatmap[int(label), bin_idx] += 1

    im = ax1.imshow(heatmap, aspect='auto', cmap='YlOrRd', interpolation='nearest',
                    extent=[0, train_size, -0.5, 9.5], origin='lower')
    ax1.set_xlabel('Training Set Index', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Class', fontsize=12, fontweight='bold')
    ax1.set_title(f'Spatial Distribution Heatmap\n({len(unique_positions)} unique training examples perturbed)', 
                  fontsize=13, fontweight='bold')
    ax1.set_yticks(range(10))
    ax1.set_yticklabels(CLASS_NAMES)
    plt.colorbar(im, ax=ax1, label='Count per bin')

    # ============================================================================
    # Plot 2: Cumulative distribution by class (top right)
    # ============================================================================
    ax2 = fig.add_subplot(gs[0, 1])

    # Plot cumulative distribution for each class
    for class_idx in range(10):
        class_mask = perturbed_class_labels == class_idx
        class_positions = np.sort(perturbed_positions[class_mask])

        if len(class_positions) > 0:
            cumulative = np.arange(1, len(class_positions) + 1)
            ax2.plot(class_positions, cumulative, label=CLASS_NAMES[class_idx], 
                     color=class_colors[class_idx], linewidth=2, alpha=0.8)

    ax2.set_xlabel('Training Set Index', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Cumulative Count', fontsize=12, fontweight='bold')
    ax2.set_title('Cumulative Distribution by Class', fontsize=13, fontweight='bold')
    ax2.legend(loc='upper left', ncol=2, fontsize=9)
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(0, train_size)

    # ============================================================================
    # Plot 3: Position density with class coloring (bottom left)
    # ============================================================================
    ax3 = fig.add_subplot(gs[1, 0])

    # Create histogram with class-based coloring
    n_bins = 80
    counts, bins, patches = ax3.hist(perturbed_positions, bins=n_bins,
                                     alpha=0.7, edgecolor='black', linewidth=0.5)

    # Color bars by predominant class in each bin
    for i in range(len(patches)):
        bin_start, bin_end = bins[i], bins[i+1]
        bin_mask = (perturbed_positions >= bin_start) & (perturbed_positions < bin_end)
        bin_labels = perturbed_class_labels[bin_mask]

        if len(bin_labels) > 0:
            unique, counts_unique = np.unique(bin_labels, return_counts=True)
            most_common_class = unique[np.argmax(counts_unique)]
            patches[i].set_facecolor(class_colors[int(most_common_class)])
            patches[i].set_alpha(0.8)

    ax3.set_xlabel('Training Set Index', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax3.set_title('Spatial Density Distribution\n(Colored by predominant class in each bin)', 
                  fontsize=13, fontweight='bold')
    ax3.grid(True, alpha=0.3, axis='y')
    ax3.set_xlim(0, train_size)

    # Add legend
    legend_handles = [mpatches.Patch(color=class_colors[i], label=CLASS_NAMES[i]) for i in range(10)]
    ax3.legend(handles=legend_handles, loc='upper right', ncol=5, fontsize=8, framealpha=0.9)

    # ============================================================================
    # Plot 4: Class distribution statistics (bottom right)
    # ============================================================================
    ax4 = fig.add_subplot(gs[1, 1])

    # Compute statistics per class
    class_stats = []
    for class_idx in range(10):
        class_mask = perturbed_class_labels == class_idx
        class_positions = perturbed_positions[class_mask]

        if len(class_positions) > 0:
            class_stats.append({
                'class': CLASS_NAMES[class_idx],
                'count': len(class_positions),
                'unique': len(np.unique(class_positions)),
                'min_pos': class_positions.min(),
                'max_pos': class_positions.max(),
                'mean_pos': class_positions.mean(),
                'spread': class_positions.max() - class_positions.min()
            })

    # Create grouped bar chart
    class_names_plot = [s['class'] for s in class_stats]
    counts = [s['count'] for s in class_stats]
    unique_counts = [s['unique'] for s in class_stats]

    x = np.arange(len(class_names_plot))
    width = 0.35

    bars1 = ax4.bar(x - width/2, counts, width, label='Total occurrences', 
                    color='steelblue', alpha=0.7, edgecolor='black')
    bars2 = ax4.bar(x + width/2, unique_counts, width, label='Unique indices',
                    color='coral', alpha=0.7, edgecolor='black')

    ax4.set_xlabel('Class', fontsize=12, fontweight='bold')
    ax4.set_ylabel('Count', fontsize=12, fontweight='bold')
    ax4.set_title('Perturbation Frequency by Class\n(Total vs Unique Training Examples)', 
                  fontsize=13, fontweight='bold')
    ax4.set_xticks(x)
    ax4.set_xticklabels(class_names_plot, rotation=45, ha='right')
    ax4.legend(fontsize=10)
    ax4.grid(True, alpha=0.3, axis='y')

    # Add count labels on bars
    for bar in bars1:
        height = bar.get_height()
        if height > 0:
            ax4.text(bar.get_x() + bar.get_width()/2., height,
                     f'{int(height)}', ha='center', va='bottom', fontsize=7)
    for bar in bars2:
        height = bar.get_height()
        if height > 0:
            ax4.text(bar.get_x() + bar.get_width()/2., height,
                     f'{int(height)}', ha='center', va='bottom', fontsize=7)

    # Main title
    fig.suptitle(
        f'Training Set Perturbation Pattern Analysis | {len(perturbed_positions)} perturbations across {len(unique_positions)} unique indices ({len(unique_positions)/train_size*100:.1f}% of training set)', 
        fontsize=14, fontweight='bold', y=0.995)

    plt.savefig(os.path.join(RESULTS_DIR, 'training_position_analysis.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # Print statistics
    sorted_positions = np.sort(perturbed_positions)
    unique_positions = np.unique(perturbed_positions)

    print(f"\nTraining Set Position Statistics:")
    print(f"  Total perturbed examples: {len(perturbed_positions)}")
    print(f"  Unique training indices: {len(unique_positions)}")
    print(f"  Training set size: {train_size}")
    print(f"  Percentage of training set perturbed: {len(unique_positions) / train_size * 100:.2f}%")
    print(f"  Position range: [{perturbed_positions.min()}, {perturbed_positions.max()}]")
    print(f"  Span: {perturbed_positions.max() - perturbed_positions.min()}")

    if len(sorted_positions) > 1:
        gaps = np.diff(sorted_positions)
        print(f"\nGap analysis (consecutive perturbed examples):")
        print(f"  Mean gap: {gaps.mean():.1f}")
        print(f"  Median gap: {np.median(gaps):.1f}")
        print(f"  Max gap: {gaps.max()}")
        print(f"  Min gap: {gaps.min()}")

        expected_gap = train_size / len(perturbed_positions)
        print(f"  Expected gap (uniform): {expected_gap:.1f}")

    print(f"\nSaved training position analysis to: {os.path.join(RESULTS_DIR, 'training_position_analysis.png')}")


In [ ]:
print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)
print(f"\nResults directory: {RESULTS_DIR}")
print(f"Total experiments completed: {len(df_log) if len(df_log) > 0 else 0}")

# Entropy and Targeted Change Analysis

This section analyzes whether infusion creates **targeted** changes to the target class probability, or whether it simply increases model entropy (uncertainty). We'll examine:

1. **Entropy changes**: Does entropy increase or decrease after infusion?
2. **Target class specificity**: Does the target class increase MORE than other classes?
3. **Probability mass flow**: Where does the probability come from when target increases?
4. **Control comparisons**: Do successful infusions behave differently from unsuccessful ones?

In [23]:
# Load all experiment results and compute entropy for each
import scipy.stats

def compute_entropy(probs):
    """Compute Shannon entropy from probability distribution"""
    # Filter out zero probabilities to avoid log(0)
    probs_nonzero = probs[probs > 0]
    return -np.sum(probs_nonzero * np.log(probs_nonzero))

if len(exp_dirs) > 0:
    print("Loading results and computing entropy...")
    
    entropy_data = []
    
    for exp_dir in exp_dirs:
        result = load_experiment_result(exp_dir)
        if result is None:
            continue
            
        # Get logits
        logits_orig = result['logits_epoch10'][0]
        logits_inf = result['logits_infused'][0]
        
        # Convert to probabilities
        probs_orig = np.exp(logits_orig - np.max(logits_orig))
        probs_orig /= probs_orig.sum()
        
        probs_inf = np.exp(logits_inf - np.max(logits_inf))
        probs_inf /= probs_inf.sum()
        
        # Compute entropies
        entropy_orig = compute_entropy(probs_orig)
        entropy_inf = compute_entropy(probs_inf)
        delta_entropy = entropy_inf - entropy_orig
        
        # Get class probabilities
        true_label = int(result['true_label'])
        target_class = int(result['target_class'])
        
        prob_target_orig = probs_orig[target_class]
        prob_target_inf = probs_inf[target_class]
        delta_p_target = prob_target_inf - prob_target_orig
        
        prob_true_orig = probs_orig[true_label]
        prob_true_inf = probs_inf[true_label]
        delta_p_true = prob_true_inf - prob_true_orig
        
        # Store all probability changes
        delta_probs_all = probs_inf - probs_orig
        
        entropy_data.append({
            'sample_idx': int(result['sample_idx']),
            'test_image_idx': int(result['test_image_idx']),
            'true_label': true_label,
            'target_class': target_class,
            'entropy_orig': entropy_orig,
            'entropy_inf': entropy_inf,
            'delta_entropy': delta_entropy,
            'prob_target_orig': prob_target_orig,
            'prob_target_inf': prob_target_inf,
            'delta_p_target': delta_p_target,
            'prob_true_orig': prob_true_orig,
            'prob_true_inf': prob_true_inf,
            'delta_p_true': delta_p_true,
            'probs_orig': probs_orig,
            'probs_inf': probs_inf,
            'delta_probs_all': delta_probs_all,
            'predicted_orig': np.argmax(probs_orig),
            'predicted_inf': np.argmax(probs_inf),
        })
    
    df_entropy = pd.DataFrame(entropy_data)
    df_entropy['success'] = df_entropy['predicted_inf'] == df_entropy['target_class']
    
    print(f"Loaded {len(df_entropy)} experiments with entropy data")
    print(f"\nEntropy statistics:")
    print(f"  Original model entropy: {df_entropy['entropy_orig'].mean():.4f} ± {df_entropy['entropy_orig'].std():.4f}")
    print(f"  Infused model entropy:  {df_entropy['entropy_inf'].mean():.4f} ± {df_entropy['entropy_inf'].std():.4f}")
    print(f"  Δ entropy (mean):       {df_entropy['delta_entropy'].mean():+.4f} ± {df_entropy['delta_entropy'].std():.4f}")
    print(f"  Δ entropy (median):     {df_entropy['delta_entropy'].median():+.4f}")
    
    # Analyze by direction
    n_increased = (df_entropy['delta_entropy'] > 0).sum()
    n_decreased = (df_entropy['delta_entropy'] < 0).sum()
    n_unchanged = (df_entropy['delta_entropy'] == 0).sum()
    
    print(f"\nEntropy change direction:")
    print(f"  Increased: {n_increased} ({n_increased/len(df_entropy)*100:.1f}%)")
    print(f"  Decreased: {n_decreased} ({n_decreased/len(df_entropy)*100:.1f}%)")
    print(f"  Unchanged: {n_unchanged} ({n_unchanged/len(df_entropy)*100:.1f}%)")
else:
    df_entropy = pd.DataFrame()
    print("No experiment directories found")

## 1. Entropy Analysis: Distribution and Correlations

First, let's examine how entropy changes after infusion and whether these changes correlate with target probability increases.

In [24]:
if len(df_entropy) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # Plot 1: Entropy change distribution
    ax = axes[0, 0]
    ax.hist(df_entropy['delta_entropy'], bins=50, alpha=0.7, edgecolor='black', color='purple')
    ax.axvline(0, color='red', linestyle='--', linewidth=2, label='No change')
    ax.axvline(df_entropy['delta_entropy'].mean(), color='green', linestyle='--', linewidth=2, 
               label=f'Mean: {df_entropy["delta_entropy"].mean():+.4f}')
    ax.set_xlabel('Δ Entropy', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title('Distribution of Entropy Changes', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Entropy vs Target Probability Change
    ax = axes[0, 1]
    ax.scatter(df_entropy['delta_entropy'], df_entropy['delta_p_target'], alpha=0.5, s=20, c='teal')
    ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax.axvline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel('Δ Entropy', fontsize=11)
    ax.set_ylabel('Δp(target)', fontsize=11)
    ax.set_title('Entropy Change vs Target Prob Change', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Add correlation
    corr = np.corrcoef(df_entropy['delta_entropy'], df_entropy['delta_p_target'])[0, 1]
    ax.text(0.05, 0.95, f'Corr: {corr:.4f}', transform=ax.transAxes,
            fontsize=10, va='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # Plot 3: Entropy vs True Class Probability Change
    ax = axes[0, 2]
    ax.scatter(df_entropy['delta_entropy'], df_entropy['delta_p_true'], alpha=0.5, s=20, c='darkgreen')
    ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax.axvline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel('Δ Entropy', fontsize=11)
    ax.set_ylabel('Δp(true)', fontsize=11)
    ax.set_title('Entropy Change vs True Class Prob Change', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Add correlation
    corr2 = np.corrcoef(df_entropy['delta_entropy'], df_entropy['delta_p_true'])[0, 1]
    ax.text(0.05, 0.95, f'Corr: {corr2:.4f}', transform=ax.transAxes,
            fontsize=10, va='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # Plot 4: Original vs Infused Entropy
    ax = axes[1, 0]
    ax.scatter(df_entropy['entropy_orig'], df_entropy['entropy_inf'], alpha=0.5, s=20, c='navy')
    min_val = min(df_entropy['entropy_orig'].min(), df_entropy['entropy_inf'].min())
    max_val = max(df_entropy['entropy_orig'].max(), df_entropy['entropy_inf'].max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, alpha=0.5, label='No change')
    ax.set_xlabel('Original Entropy', fontsize=11)
    ax.set_ylabel('Infused Entropy', fontsize=11)
    ax.set_title('Original vs Infused Entropy', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 5: Entropy change by success/failure
    ax = axes[1, 1]
    success_entropy = df_entropy[df_entropy['success'] == True]['delta_entropy']
    failure_entropy = df_entropy[df_entropy['success'] == False]['delta_entropy']
    ax.hist([success_entropy, failure_entropy], bins=30, alpha=0.7, edgecolor='black',
            label=[f'Success (n={len(success_entropy)})', f'Failure (n={len(failure_entropy)})'],
            color=['green', 'red'])
    ax.axvline(0, color='black', linestyle='--', linewidth=2, alpha=0.5)
    ax.set_xlabel('Δ Entropy', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title('Entropy Change: Successful vs Failed Infusions', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 6: Box plot comparison
    ax = axes[1, 2]
    ax.boxplot([df_entropy['entropy_orig'], df_entropy['entropy_inf']], 
                labels=['Original', 'Infused'], patch_artist=True)
    ax.set_ylabel('Entropy', fontsize=11)
    ax.set_title('Entropy Distribution: Original vs Infused', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'entropy_analysis.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nSaved entropy analysis to: {os.path.join(RESULTS_DIR, 'entropy_analysis.png')}")

## 2. Statistical Significance Tests

Now we'll perform statistical tests to demonstrate that the target probability increase is **statistically significant** and **independent** of entropy changes. We'll use three complementary measures:

1. **Paired t-test**: Tests if target probability significantly increased after infusion
2. **Partial correlation**: Tests if Δp(target) is significant after controlling for entropy changes
3. **Linear regression**: Tests if Δp(target) is explained by targeting, not just entropy

In [25]:
if len(df_entropy) > 0:
    from scipy import stats
    
    print("="*80)
    print("STATISTICAL SIGNIFICANCE TESTS")
    print("="*80)
    
    # ============================================================================
    # TEST 1: Paired t-test - Is target probability increase significant?
    # ============================================================================
    print("\n" + "="*80)
    print("TEST 1: PAIRED T-TEST")
    print("="*80)
    print("\nNull Hypothesis: Target probability does not increase after infusion")
    print("Alternative Hypothesis: Target probability increases after infusion\n")
    
    # Perform one-tailed paired t-test
    t_stat, p_value_two_tailed = stats.ttest_rel(
        df_entropy['prob_target_inf'], 
        df_entropy['prob_target_orig']
    )
    p_value_one_tailed = p_value_two_tailed / 2  # One-tailed test
    
    print(f"Sample size: {len(df_entropy)}")
    print(f"Mean target probability (original): {df_entropy['prob_target_orig'].mean():.6f}")
    print(f"Mean target probability (infused):  {df_entropy['prob_target_inf'].mean():.6f}")
    print(f"Mean increase: {df_entropy['delta_p_target'].mean():.6f}")
    print(f"\nT-statistic: {t_stat:.4f}")
    print(f"P-value (one-tailed): {p_value_one_tailed:.2e}")
    
    if p_value_one_tailed < 0.001:
        print(f"\n✓ RESULT: Target probability increase is HIGHLY SIGNIFICANT (p < 0.001)")
        print(f"  The probability that this increase occurred by chance is less than 0.1%")
    elif p_value_one_tailed < 0.01:
        print(f"\n✓ RESULT: Target probability increase is VERY SIGNIFICANT (p < 0.01)")
    elif p_value_one_tailed < 0.05:
        print(f"\n✓ RESULT: Target probability increase is SIGNIFICANT (p < 0.05)")
    else:
        print(f"\n✗ RESULT: Target probability increase is NOT significant (p = {p_value_one_tailed:.4f})")
    
    # Effect size (Cohen's d)
    cohens_d = df_entropy['delta_p_target'].mean() / df_entropy['delta_p_target'].std()
    print(f"\nEffect size (Cohen's d): {cohens_d:.4f}")
    if abs(cohens_d) > 0.8:
        print("  (Large effect)")
    elif abs(cohens_d) > 0.5:
        print("  (Medium effect)")
    elif abs(cohens_d) > 0.2:
        print("  (Small effect)")
    else:
        print("  (Negligible effect)")
    
    # ============================================================================
    # TEST 2: Partial Correlation - Is Δp(target) significant after controlling for entropy?
    # ============================================================================
    print("\n\n" + "="*80)
    print("TEST 2: PARTIAL CORRELATION ANALYSIS")
    print("="*80)
    print("\nQuestion: Is the target probability increase still significant")
    print("after controlling for entropy changes?\n")
    
    # Compute partial correlation between Δp(target) and a constant (representing the infusion effect)
    # while controlling for Δentropy
    from scipy.stats import pearsonr
    
    # First, let's see the zero-order correlations
    corr_target_entropy, p_corr_te = pearsonr(df_entropy['delta_p_target'], df_entropy['delta_entropy'])
    
    print(f"Zero-order correlation between Δp(target) and Δentropy: {corr_target_entropy:.4f}")
    print(f"P-value: {p_corr_te:.4f}")
    
    if abs(corr_target_entropy) < 0.3:
        print("\n✓ INTERPRETATION: Weak correlation indicates target probability changes")
        print("  are largely INDEPENDENT of entropy changes")
    else:
        print(f"\nNote: Moderate correlation suggests some relationship with entropy")
    
    # Partial correlation using linear regression residuals
    # We want to test if Δp(target) has variance unexplained by Δentropy
    from sklearn.linear_model import LinearRegression
    
    # Regress Δp(target) on Δentropy
    X_entropy = df_entropy[['delta_entropy']].values
    y_delta_p_target = df_entropy['delta_p_target'].values
    
    model_entropy = LinearRegression()
    model_entropy.fit(X_entropy, y_delta_p_target)
    residuals = y_delta_p_target - model_entropy.predict(X_entropy)
    
    # Test if residuals (part of Δp(target) unexplained by entropy) are significantly > 0
    t_stat_residual, p_value_residual = stats.ttest_1samp(residuals, 0)
    
    print(f"\n--- Partial Correlation Test ---")
    print(f"Variance in Δp(target) explained by Δentropy: {model_entropy.score(X_entropy, y_delta_p_target):.2%}")
    print(f"Variance unexplained by entropy: {1 - model_entropy.score(X_entropy, y_delta_p_target):.2%}")
    print(f"\nMean of residuals (Δp after removing entropy effect): {residuals.mean():.6f}")
    print(f"T-statistic for residuals: {t_stat_residual:.4f}")
    print(f"P-value: {p_value_residual:.2e}")
    
    if p_value_residual < 0.001:
        print(f"\n✓ RESULT: Target probability increase is HIGHLY SIGNIFICANT")
        print(f"  even after controlling for entropy changes (p < 0.001)")
        print(f"\n  INTERPRETATION: The infusion creates targeted changes that are")
        print(f"  independent of entropy effects!")
    elif p_value_residual < 0.05:
        print(f"\n✓ RESULT: Target probability increase is SIGNIFICANT")
        print(f"  even after controlling for entropy changes (p < 0.05)")
    else:
        print(f"\n✗ RESULT: Target probability increase is NOT significant")
        print(f"  after controlling for entropy (p = {p_value_residual:.4f})")
    
    # ============================================================================
    # TEST 3: Multiple Regression - What predicts Δp(target)?
    # ============================================================================
    print("\n\n" + "="*80)
    print("TEST 3: MULTIPLE REGRESSION ANALYSIS")
    print("="*80)
    print("\nQuestion: What factors predict Δp(target)?")
    print("We'll test if targeting effects exist beyond entropy changes.\n")
    
    # Create features
    # We'll use: Δentropy, original target prob, and a 'targeting indicator'
    # The targeting indicator is whether the target differs from true class
    df_entropy['is_targeted'] = (df_entropy['target_class'] != df_entropy['true_label']).astype(int)
    
    from sklearn.linear_model import LinearRegression
    from scipy import stats as scipy_stats
    
    # Model: Δp(target) ~ Δentropy + p_target_orig + is_targeted
    X = df_entropy[['delta_entropy', 'prob_target_orig', 'is_targeted']].values
    y = df_entropy['delta_p_target'].values
    
    # Fit regression
    reg_model = LinearRegression()
    reg_model.fit(X, y)
    
    # Predict and compute residuals
    y_pred = reg_model.predict(X)
    residuals_reg = y - y_pred
    
    # Compute standard errors for coefficients (for t-tests)
    n = len(y)
    k = X.shape[1]
    mse = np.sum(residuals_reg**2) / (n - k - 1)
    
    # Variance-covariance matrix
    X_with_intercept = np.column_stack([np.ones(n), X])
    var_covar = mse * np.linalg.inv(X_with_intercept.T @ X_with_intercept)
    se = np.sqrt(np.diag(var_covar))
    
    # Coefficients with standard errors
    coefs = np.concatenate([[reg_model.intercept_], reg_model.coef_])
    t_stats_reg = coefs / se
    p_values_reg = 2 * (1 - scipy_stats.t.cdf(np.abs(t_stats_reg), n - k - 1))
    
    print(f"R² = {reg_model.score(X, y):.4f}")
    print(f"  ({reg_model.score(X, y)*100:.2f}% of variance in Δp(target) explained)\n")
    
    print("Regression Coefficients:")
    print(f"  Intercept:        {reg_model.intercept_:+.6f}  (SE={se[0]:.6f}, p={p_values_reg[0]:.4f})")
    print(f"  Δentropy:         {reg_model.coef_[0]:+.6f}  (SE={se[1]:.6f}, p={p_values_reg[1]:.4f})")
    print(f"  p_target_orig:    {reg_model.coef_[1]:+.6f}  (SE={se[2]:.6f}, p={p_values_reg[2]:.4f})")
    print(f"  is_targeted:      {reg_model.coef_[2]:+.6f}  (SE={se[3]:.6f}, p={p_values_reg[3]:.4f})")
    
    print("\n--- Interpretation ---")
    
    # Interpret Δentropy coefficient
    if p_values_reg[1] < 0.05:
        print(f"\n• Δentropy: SIGNIFICANT (p={p_values_reg[1]:.4f})")
        print(f"  Each unit increase in entropy changes target prob by {reg_model.coef_[0]:+.4f}")
    else:
        print(f"\n• Δentropy: NOT significant (p={p_values_reg[1]:.4f})")
        print(f"  Entropy changes do NOT strongly predict target probability changes")
    
    # Interpret original target probability
    if p_values_reg[2] < 0.05:
        print(f"\n• Original target probability: SIGNIFICANT (p={p_values_reg[2]:.4f})")
        direction = "harder" if reg_model.coef_[1] < 0 else "easier"
        print(f"  Higher original probabilities make infusion {direction}")
    else:
        print(f"\n• Original target probability: NOT significant (p={p_values_reg[2]:.4f})")
    
    # Interpret targeting indicator (THIS IS KEY!)
    if p_values_reg[3] < 0.001:
        print(f"\n• Targeting indicator: HIGHLY SIGNIFICANT (p < 0.001)")
        print(f"  Being a targeted class (≠ true class) adds {reg_model.coef_[2]:+.4f} to Δp(target)")
        print(f"\n  ✓ STRONG EVIDENCE: Infusion creates targeted changes beyond entropy!")
    elif p_values_reg[3] < 0.05:
        print(f"\n• Targeting indicator: SIGNIFICANT (p={p_values_reg[3]:.4f})")
        print(f"  ✓ Evidence of targeted changes beyond entropy")
    else:
        print(f"\n• Targeting indicator: NOT significant (p={p_values_reg[3]:.4f})")
    
    # ============================================================================
    # SUMMARY
    # ============================================================================
    print("\n\n" + "="*80)
    print("OVERALL SUMMARY")
    print("="*80)
    
    print("\nThe three statistical tests provide complementary evidence:\n")
    
    # Test 1 summary
    if p_value_one_tailed < 0.001:
        print("✓ TEST 1: Target probability increases are HIGHLY significant")
    else:
        print("? TEST 1: Target probability increases are not strongly significant")
    
    # Test 2 summary  
    if p_value_residual < 0.001:
        print("✓ TEST 2: Increases remain HIGHLY significant after controlling for entropy")
    else:
        print("? TEST 2: Significance weakens when controlling for entropy")
    
    # Test 3 summary
    if p_values_reg[3] < 0.001:
        print("✓ TEST 3: Targeting effects are HIGHLY significant in regression model")
    else:
        print("? TEST 3: Targeting effects are not strongly significant")
    
    print("\n" + "-"*80)
    
    # Overall conclusion
    all_significant = (p_value_one_tailed < 0.001 and 
                      p_value_residual < 0.001 and 
                      p_values_reg[3] < 0.001)
    
    if all_significant:
        print("\n🎯 CONCLUSION: Strong statistical evidence that infusion creates")
        print("   TARGETED changes to the target class probability that are")
        print("   INDEPENDENT of entropy changes!")
        print("\n   The probability increases are:")
        print("   • Statistically significant (p < 0.001)")
        print("   • Not explained by entropy changes alone")
        print("   • Specific to the targeted class")
    else:
        print("\n📊 CONCLUSION: Evidence suggests targeted changes, but some tests")
        print("   show weaker significance. Review individual test results above.")
    
    print("\n" + "="*80)

## 3. Extended Statistical Tests

Since initial tests were inconclusive, we'll perform more rigorous analyses:

1. **Wilcoxon Signed-Rank Test**: Non-parametric alternative to t-test (doesn't assume normality)
2. **Bootstrap Confidence Intervals**: Robust estimate of effect size with confidence bounds
3. **Comparison with Other Classes**: Test if target class increases MORE than non-target classes
4. **Permutation Test**: Test significance by comparing to random permutations
5. **Effect Size Decomposition**: Separate entropy effects from targeting effects

In [26]:
if len(df_entropy) > 0:
    from scipy import stats
    import numpy as np
    from sklearn.utils import resample
    
    print("="*80)
    print("EXTENDED STATISTICAL TESTS")
    print("="*80)
    
    # ============================================================================
    # TEST 4: Wilcoxon Signed-Rank Test (Non-parametric)
    # ============================================================================
    print("\n" + "="*80)
    print("TEST 4: WILCOXON SIGNED-RANK TEST")
    print("="*80)
    print("\nThis is a non-parametric test that doesn't assume normal distribution.")
    print("It tests if the median difference is significantly different from zero.\n")
    
    # Wilcoxon signed-rank test
    statistic, p_value_wilcoxon = stats.wilcoxon(
        df_entropy['prob_target_inf'], 
        df_entropy['prob_target_orig'],
        alternative='greater'
    )
    
    median_diff = df_entropy['delta_p_target'].median()
    
    print(f"Sample size: {len(df_entropy)}")
    print(f"Median Δp(target): {median_diff:.6f}")
    print(f"Wilcoxon statistic: {statistic:.1f}")
    print(f"P-value (one-tailed): {p_value_wilcoxon:.2e}")
    
    if p_value_wilcoxon < 0.001:
        print(f"\n✓ RESULT: HIGHLY SIGNIFICANT (p < 0.001)")
        print(f"  The median target probability increase is statistically significant")
    elif p_value_wilcoxon < 0.01:
        print(f"\n✓ RESULT: VERY SIGNIFICANT (p < 0.01)")
    elif p_value_wilcoxon < 0.05:
        print(f"\n✓ RESULT: SIGNIFICANT (p < 0.05)")
    else:
        print(f"\n✗ RESULT: NOT significant (p = {p_value_wilcoxon:.4f})")
    
    # ============================================================================
    # TEST 5: Bootstrap Confidence Intervals
    # ============================================================================
    print("\n\n" + "="*80)
    print("TEST 5: BOOTSTRAP CONFIDENCE INTERVALS")
    print("="*80)
    print("\nBootstrapping provides robust confidence intervals without")
    print("assumptions about the data distribution.\n")
    
    n_bootstrap = 10000
    np.random.seed(42)
    
    bootstrap_means = []
    bootstrap_medians = []
    
    for _ in range(n_bootstrap):
        sample = resample(df_entropy['delta_p_target'].values)
        bootstrap_means.append(sample.mean())
        bootstrap_medians.append(np.median(sample))
    
    bootstrap_means = np.array(bootstrap_means)
    bootstrap_medians = np.array(bootstrap_medians)
    
    # 95% confidence intervals
    ci_mean_lower = np.percentile(bootstrap_means, 2.5)
    ci_mean_upper = np.percentile(bootstrap_means, 97.5)
    ci_median_lower = np.percentile(bootstrap_medians, 2.5)
    ci_median_upper = np.percentile(bootstrap_medians, 97.5)
    
    print(f"Number of bootstrap samples: {n_bootstrap:,}")
    print(f"\nMean Δp(target):")
    print(f"  Point estimate: {df_entropy['delta_p_target'].mean():.6f}")
    print(f"  95% CI: [{ci_mean_lower:.6f}, {ci_mean_upper:.6f}]")
    
    print(f"\nMedian Δp(target):")
    print(f"  Point estimate: {df_entropy['delta_p_target'].median():.6f}")
    print(f"  95% CI: [{ci_median_lower:.6f}, {ci_median_upper:.6f}]")
    
    if ci_mean_lower > 0:
        print(f"\n✓ RESULT: SIGNIFICANT - 95% CI excludes zero")
        print(f"  We can be 95% confident the true mean increase is positive")
    else:
        print(f"\n✗ RESULT: NOT conclusive - 95% CI includes zero")
    
    # Probability that effect is positive
    prob_positive = (bootstrap_means > 0).mean()
    print(f"\nProbability that mean Δp(target) > 0: {prob_positive:.1%}")
    
    # ============================================================================
    # TEST 6: Target vs Non-Target Class Comparison
    # ============================================================================
    print("\n\n" + "="*80)
    print("TEST 6: TARGET VS NON-TARGET CLASS COMPARISON")
    print("="*80)
    print("\nKey Question: Does the TARGET class increase MORE than other classes?")
    print("This directly tests if changes are targeted vs uniform.\n")
    
    # For each experiment, compute average Δp for non-target classes
    delta_p_target_list = []
    delta_p_nontarget_list = []
    delta_p_other_excluding_true_list = []
    
    for idx, row in df_entropy.iterrows():
        delta_probs = row['delta_probs_all']
        target_class = row['target_class']
        true_label = row['true_label']
        
        # Target class change
        delta_p_target = delta_probs[target_class]
        delta_p_target_list.append(delta_p_target)
        
        # Average change for all non-target classes
        non_target_mask = np.ones(10, dtype=bool)
        non_target_mask[target_class] = False
        delta_p_nontarget = delta_probs[non_target_mask].mean()
        delta_p_nontarget_list.append(delta_p_nontarget)
        
        # Average change for classes excluding both target and true
        other_mask = np.ones(10, dtype=bool)
        other_mask[target_class] = False
        other_mask[true_label] = False
        if other_mask.sum() > 0:
            delta_p_other = delta_probs[other_mask].mean()
            delta_p_other_excluding_true_list.append(delta_p_other)
    
    delta_p_target_arr = np.array(delta_p_target_list)
    delta_p_nontarget_arr = np.array(delta_p_nontarget_list)
    delta_p_other_arr = np.array(delta_p_other_excluding_true_list)
    
    # Paired comparison: target vs average non-target
    difference = delta_p_target_arr - delta_p_nontarget_arr
    
    print(f"Mean Δp(target):     {delta_p_target_arr.mean():.6f}")
    print(f"Mean Δp(non-target): {delta_p_nontarget_arr.mean():.6f}")
    print(f"Mean difference:     {difference.mean():.6f}")
    
    # Test if difference is significant
    t_stat_diff, p_value_diff = stats.ttest_rel(delta_p_target_arr, delta_p_nontarget_arr)
    
    print(f"\nPaired t-test (target vs non-target):")
    print(f"  T-statistic: {t_stat_diff:.4f}")
    print(f"  P-value: {p_value_diff:.2e}")
    
    if p_value_diff < 0.001:
        print(f"\n✓ RESULT: HIGHLY SIGNIFICANT (p < 0.001)")
        print(f"  Target class increases significantly MORE than other classes")
        print(f"  This is strong evidence of TARGETED changes!")
    elif p_value_diff < 0.01:
        print(f"\n✓ RESULT: VERY SIGNIFICANT (p < 0.01)")
        print(f"  Target class increases more than other classes")
    elif p_value_diff < 0.05:
        print(f"\n✓ RESULT: SIGNIFICANT (p < 0.05)")
    else:
        print(f"\n✗ RESULT: NOT significant (p = {p_value_diff:.4f})")
        print(f"  Target class doesn't increase significantly more than others")
    
    # Effect size
    cohens_d_diff = difference.mean() / difference.std()
    print(f"\nEffect size (Cohen's d): {cohens_d_diff:.4f}")
    
    # Also test vs classes excluding both target and true
    if len(delta_p_other_arr) > 0:
        difference2 = delta_p_target_arr[:len(delta_p_other_arr)] - delta_p_other_arr
        print(f"\n--- Comparison excluding true class ---")
        print(f"Mean Δp(other classes): {delta_p_other_arr.mean():.6f}")
        print(f"Mean difference (target - other): {difference2.mean():.6f}")
        
        t_stat_diff2, p_value_diff2 = stats.ttest_1samp(difference2, 0)
        print(f"T-statistic: {t_stat_diff2:.4f}")
        print(f"P-value: {p_value_diff2:.2e}")
        
        if p_value_diff2 < 0.001:
            print(f"✓ Target increases significantly more (p < 0.001)")
    
    # ============================================================================
    # TEST 7: Permutation Test
    # ============================================================================
    print("\n\n" + "="*80)
    print("TEST 7: PERMUTATION TEST")
    print("="*80)
    print("\nThis tests if the observed difference could occur by chance")
    print("by randomly permuting which class is 'targeted'.\n")
    
    n_permutations = 10000
    np.random.seed(42)
    
    # Observed statistic: mean difference (target - non-target)
    observed_stat = difference.mean()
    
    print(f"Computing {n_permutations:,} permutations...")
    
    permuted_stats = []
    for _ in range(n_permutations):
        # For each experiment, randomly select which class to treat as "target"
        permuted_diffs = []
        for idx, row in df_entropy.iterrows():
            delta_probs = row['delta_probs_all']
            
            # Randomly pick a "target" class
            random_target = np.random.randint(0, 10)
            
            # Compute difference
            non_target_mask = np.ones(10, dtype=bool)
            non_target_mask[random_target] = False
            
            diff = delta_probs[random_target] - delta_probs[non_target_mask].mean()
            permuted_diffs.append(diff)
        
        permuted_stats.append(np.mean(permuted_diffs))
    
    permuted_stats = np.array(permuted_stats)
    
    # P-value: proportion of permutations with stat >= observed
    p_value_perm = (permuted_stats >= observed_stat).mean()
    
    print(f"\nObserved mean difference: {observed_stat:.6f}")
    print(f"Mean of permuted differences: {permuted_stats.mean():.6f}")
    print(f"Std of permuted differences: {permuted_stats.std():.6f}")
    print(f"\nP-value (permutation test): {p_value_perm:.4f}")
    
    if p_value_perm < 0.001:
        print(f"\n✓ RESULT: HIGHLY SIGNIFICANT (p < 0.001)")
        print(f"  Less than 0.1% of random permutations show this effect")
        print(f"  Strong evidence that targeting is NOT random!")
    elif p_value_perm < 0.01:
        print(f"\n✓ RESULT: VERY SIGNIFICANT (p < 0.01)")
    elif p_value_perm < 0.05:
        print(f"\n✓ RESULT: SIGNIFICANT (p < 0.05)")
    else:
        print(f"\n✗ RESULT: NOT significant (p = {p_value_perm:.4f})")
        print(f"  The observed effect could occur by chance")
    
    # Show distribution
    print(f"\nPercentiles of permutation distribution:")
    print(f"  95th: {np.percentile(permuted_stats, 95):.6f}")
    print(f"  99th: {np.percentile(permuted_stats, 99):.6f}")
    print(f"  99.9th: {np.percentile(permuted_stats, 99.9):.6f}")
    print(f"  Observed: {observed_stat:.6f}")
    
    # ============================================================================
    # TEST 8: Effect Size Decomposition
    # ============================================================================
    print("\n\n" + "="*80)
    print("TEST 8: EFFECT SIZE DECOMPOSITION")
    print("="*80)
    print("\nDecompose Δp(target) into components:")
    print("  1. Entropy-related changes")
    print("  2. Residual (targeted) changes\n")
    
    from sklearn.linear_model import LinearRegression
    
    # Regress Δp(target) on Δentropy
    X_entropy = df_entropy[['delta_entropy']].values
    y_target = df_entropy['delta_p_target'].values
    
    X_entropy_centered = df_entropy['delta_entropy'] - df_entropy['delta_entropy'].mean()
    model = LinearRegression()
    model.fit(X_entropy_centered.values.reshape(-1,1), y_target)

    
    # Predicted values (entropy component)
    entropy_component = model.predict(X_entropy)
    # Residuals (targeting component)
    targeting_component = y_target - entropy_component
    
    print(f"Variance decomposition:")
    print(f"  Total variance in Δp(target): {y_target.var():.8f}")
    print(f"  Variance explained by entropy: {entropy_component.var():.8f} ({entropy_component.var()/y_target.var()*100:.2f}%)")
    print(f"  Residual variance (targeting): {targeting_component.var():.8f} ({targeting_component.var()/y_target.var()*100:.2f}%)")
    
    print(f"\nMean decomposition:")
    print(f"  Total mean Δp(target):   {y_target.mean():.6f}")
    print(f"  Entropy component:       {entropy_component.mean():.6f}")
    print(f"  Targeting component:     {targeting_component.mean():.6f}")
    
    # Test if targeting component is significant
    t_stat_target_comp, p_value_target_comp = stats.ttest_1samp(targeting_component, 0)
    
    print(f"\nTest if targeting component ≠ 0:")
    print(f"  T-statistic: {t_stat_target_comp:.4f}")
    print(f"  P-value: {p_value_target_comp:.2e}")
    
    if p_value_target_comp < 0.001:
        print(f"\n✓ RESULT: HIGHLY SIGNIFICANT (p < 0.001)")
        print(f"  After removing entropy effects, a significant targeting effect remains")
    elif p_value_target_comp < 0.05:
        print(f"\n✓ RESULT: SIGNIFICANT (p < 0.05)")
    else:
        print(f"\n✗ RESULT: NOT significant (p = {p_value_target_comp:.4f})")
    
    # Ratio of targeting to entropy
    if abs(entropy_component.mean()) > 1e-6:
        ratio = targeting_component.mean() / entropy_component.mean()
        print(f"\nRatio of targeting to entropy effects: {ratio:.2f}")
        if abs(ratio) > 2:
            print(f"  Targeting effects are {abs(ratio):.1f}× larger than entropy effects")
    
    # ============================================================================
    # TEST 9: Sign Test
    # ============================================================================
    print("\n\n" + "="*80)
    print("TEST 9: SIGN TEST")
    print("="*80)
    print("\nSimplest possible test: Do MORE cases increase than decrease?")
    print("(This is the most robust to outliers and distributions)\n")
    
    n_positive = (df_entropy['delta_p_target'] > 0).sum()
    n_negative = (df_entropy['delta_p_target'] < 0).sum()
    n_zero = (df_entropy['delta_p_target'] == 0).sum()
    n_total = n_positive + n_negative
    
    print(f"Cases where Δp(target) > 0: {n_positive} ({n_positive/len(df_entropy)*100:.1f}%)")
    print(f"Cases where Δp(target) < 0: {n_negative} ({n_negative/len(df_entropy)*100:.1f}%)")
    print(f"Cases where Δp(target) = 0: {n_zero}")
    
    # Binomial test: under null hypothesis, P(positive) = 0.5
    try:
        # Scipy >=1.7.0: stats.binomtest
        p_value_sign = stats.binomtest(n_positive, n_total, 0.5, alternative='greater').pvalue
    except AttributeError:
        # Fallback: use scipy.stats.binom_test (deprecated in newer scipy)
        p_value_sign = stats.binom_test(n_positive, n_total, 0.5, alternative='greater')
    
    print(f"\nBinomial test (H0: P(positive) = 0.5):")
    print(f"  P-value: {p_value_sign:.2e}")
    
    if p_value_sign < 0.001:
        print(f"\n✓ RESULT: HIGHLY SIGNIFICANT (p < 0.001)")
        print(f"  Vastly more increases than expected by chance")
    elif p_value_sign < 0.01:
        print(f"\n✓ RESULT: VERY SIGNIFICANT (p < 0.01)")
    elif p_value_sign < 0.05:
        print(f"\n✓ RESULT: SIGNIFICANT (p < 0.05)")
    else:
        print(f"\n✗ RESULT: NOT significant (p = {p_value_sign:.4f})")
    
    # ============================================================================
    # COMPREHENSIVE SUMMARY
    # ============================================================================
    print("\n\n" + "="*80)
    print("COMPREHENSIVE SUMMARY OF ALL TESTS")
    print("="*80)
    
    test_results = {
        'Wilcoxon Test': p_value_wilcoxon,
        'Bootstrap CI': 0.0 if ci_mean_lower > 0 else 1.0,
        'Target vs Non-Target': p_value_diff,
        'Permutation Test': p_value_perm,
        'Targeting Component': p_value_target_comp,
        'Sign Test': p_value_sign,
    }
    
    print("\nP-values for all tests:")
    for test_name, p_val in test_results.items():
        sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
        print(f"  {test_name:.<30} p = {p_val:.4f} {sig}")
    
    print("\n  Significance: *** p<0.001, ** p<0.01, * p<0.05, ns=not significant")
    
    # Count significant tests
    n_significant_001 = sum(1 for p in test_results.values() if p < 0.001)
    n_significant_01 = sum(1 for p in test_results.values() if p < 0.01)
    n_significant_05 = sum(1 for p in test_results.values() if p < 0.05)
    n_total_tests = len(test_results)
    
    print(f"\n{n_significant_001}/{n_total_tests} tests significant at p < 0.001")
    print(f"{n_significant_01}/{n_total_tests} tests significant at p < 0.01")
    print(f"{n_significant_05}/{n_total_tests} tests significant at p < 0.05")
    
    print("\n" + "="*80)
    print("FINAL CONCLUSION")
    print("="*80)
    
    if n_significant_001 >= 4:
        print("\n🎯 STRONG EVIDENCE FOR TARGETED INFUSION:")
        print(f"   {n_significant_001} out of {n_total_tests} tests show p < 0.001")
        print("\n   Key findings:")
        print(f"   • Target probability increases significantly (median: {median_diff:+.6f})")
        print(f"   • Target class increases MORE than non-target classes")
        print(f"   • Effects remain after controlling for entropy")
        print(f"   • {n_positive} of {n_total} cases ({n_positive/n_total*100:.1f}%) show increases")
    elif n_significant_05 >= 4:
        print("\n📊 MODERATE EVIDENCE FOR TARGETED INFUSION:")
        print(f"   {n_significant_05} out of {n_total_tests} tests show p < 0.05")
        print("\n   The data suggests targeted effects, though with some variability")
    else:
        print("\n❓ INCONCLUSIVE RESULTS:")
        print(f"   Only {n_significant_05} out of {n_total_tests} tests significant")
        print("\n   Possible reasons:")
        print("   • Effect size may be small")
        print("   • High variance in responses")
        print("   • Sample size may be insufficient")
        print("   • Entropy effects may dominate")
    
    print("\n" + "="*80)

In [27]:
# ============================================================================
# VISUALIZATION: Δp(target) vs Δentropy
# ============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

# Center entropy for clarity
delta_entropy_centered = df_entropy['delta_entropy'] - df_entropy['delta_entropy'].mean()
y_target = df_entropy['delta_p_target']

# Fit regression line
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(delta_entropy_centered.values.reshape(-1,1), y_target)
pred_line = model.predict(delta_entropy_centered.values.reshape(-1,1))
residuals = y_target - pred_line

# Create the plot
plt.figure(figsize=(8,6))
sns.scatterplot(x=delta_entropy_centered, y=y_target, alpha=0.6, label="Data points")
sns.lineplot(x=delta_entropy_centered, y=pred_line, color="red", label="Fitted line (entropy component)")

# Add residual lines for a few random points
sample_idx = np.random.choice(len(df_entropy), size=min(30, len(df_entropy)), replace=False)
for i in sample_idx:
    plt.plot(
        [delta_entropy_centered.iloc[i], delta_entropy_centered.iloc[i]],
        [pred_line[i], y_target.iloc[i]],
        color="gray", alpha=0.4, linewidth=1
    )

# Mark the intercept visually
plt.axhline(model.intercept_, color="blue", linestyle="--", label=f"Intercept (β₀ = {model.intercept_:.3f})")
plt.axhline(0, color="black", linewidth=0.8)

plt.xlabel("ΔEntropy (centered)")
plt.ylabel("Δp(Target)")
plt.title("Regression of Δp(Target) on ΔEntropy\nShowing entropy component and residuals")
plt.legend()
plt.tight_layout()
plt.show()

# Show the mean of residuals to confirm it's ~0
print(f"Mean of residuals (targeting component): {residuals.mean():.6f}")
print("→ This is always ~0 by design of OLS regression.")


In [28]:
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import numpy as np
from sklearn.linear_model import LinearRegression

# Center entropy
delta_entropy_centered = df_entropy['delta_entropy'] - df_entropy['delta_entropy'].mean()
y_target = df_entropy['delta_p_target']

# Fit regression
model = LinearRegression()
model.fit(delta_entropy_centered.values.reshape(-1,1), y_target)
pred_line = model.predict(delta_entropy_centered.values.reshape(-1,1))

# Prepare sorted values for plotting fitted line
delta_entropy_sorted = np.sort(delta_entropy_centered.values)
fitted_sorted = model.predict(delta_entropy_sorted.reshape(-1, 1))

# Create combined figure: left scatter, right bar
fig = make_subplots(rows=1, cols=2, 
                    column_widths=[0.6, 0.4],
                    subplot_titles=("Δp(Target) vs ΔEntropy", "Probability Distribution"))

# --- LEFT PANEL: Regression scatter + line ---
fig.add_trace(
    go.Scatter(
        x=delta_entropy_centered,
        y=y_target,
        mode='markers',
        name='Data points',
        marker=dict(size=6, color='steelblue', opacity=0.7),
        customdata=np.arange(len(df_entropy)),  # keep index to identify clicked point
        hovertemplate="Index %{customdata}<br>ΔEntropy=%{x:.3f}<br>Δp(Target)=%{y:.3f}"
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=delta_entropy_sorted,
        y=fitted_sorted,
        mode='lines',
        name='Fitted line',
        line=dict(color='red')
    ),
    row=1, col=1
)

fig.update_xaxes(title_text="ΔEntropy (centered)", row=1, col=1)
fig.update_yaxes(title_text="Δp(Target)", row=1, col=1)

# --- RIGHT PANEL: placeholder bar chart ---
CLASS_NAMES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
empty_bar = go.Bar(x=CLASS_NAMES, y=[0]*10, marker_color='coral', name="Infused")
fig.add_trace(empty_bar, row=1, col=2)
fig.update_yaxes(title_text="Probability", row=1, col=2)

# --- Make interactive callback in Jupyter ---
from IPython.display import display
import ipywidgets as widgets

out = widgets.Output()

def update_bar(trace, points, selector):
    with out:
        out.clear_output(wait=True)
        if not points.point_inds:
            return
        idx = points.point_inds[0]
        row = df_entropy.iloc[idx]

        probs_epoch10 = row['probs_epoch10']  # assumes these columns exist
        probs_infused = row['probs_infused']
        true_label = row['true_label']
        target_class = row['target_class']

        bar1 = go.Bar(x=CLASS_NAMES, y=probs_epoch10, name='Original (epoch 10)', marker_color='steelblue')
        bar2 = go.Bar(x=CLASS_NAMES, y=probs_infused, name='Infused', marker_color='coral')
        vline_true = go.Scatter(x=[CLASS_NAMES[true_label]], y=[max(probs_infused)], mode='markers+text',
                                marker=dict(color='green', size=12), text=['True'], textposition='top center')
        vline_target = go.Scatter(x=[CLASS_NAMES[target_class]], y=[max(probs_infused)], mode='markers+text',
                                  marker=dict(color='red', size=12), text=['Target'], textposition='top center')

        bar_fig = go.Figure(data=[bar1, bar2, vline_true, vline_target])
        bar_fig.update_layout(title_text=f"Probability Distribution (Example {idx})",
                              xaxis_title="Class", yaxis_title="Probability",
                              barmode='group', width=600, height=400)
        bar_fig.show()

# Attach click event
fig.data[0].on_click(update_bar)

display(fig, out)


In [29]:
# Visualization of key statistical findings
if len(df_entropy) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # Plot 1: Target vs Non-Target comparison
    ax = axes[0, 0]
    positions = [1, 2]
    bp = ax.boxplot([delta_p_target_arr, delta_p_nontarget_arr], 
                     positions=positions, labels=['Target', 'Non-Target'],
                     patch_artist=True, widths=0.6)
    bp['boxes'][0].set_facecolor('coral')
    bp['boxes'][1].set_facecolor('lightblue')
    ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_ylabel('Δp', fontsize=11)
    ax.set_title(f'Target vs Non-Target Classes\n(p = {p_value_diff:.2e})', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Plot 2: Histogram of differences
    ax = axes[0, 1]
    ax.hist(difference, bins=50, alpha=0.7, edgecolor='black', color='green')
    ax.axvline(0, color='red', linestyle='--', linewidth=2, label='No difference')
    ax.axvline(difference.mean(), color='blue', linestyle='--', linewidth=2, 
               label=f'Mean: {difference.mean():.4f}')
    ax.set_xlabel('Δp(target) - Δp(non-target)', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title('Distribution of Target - Non-Target', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 3: Permutation test visualization [y axis log]
    ax = axes[0, 2]
    # Plot permuted stats histogram (null)
    n_perm, bins_perm, _ = ax.hist(permuted_stats, bins=50, alpha=0.7, edgecolor='black', color='lightgray', label='Permutations')
    # Also plot the actual observed differences for the targets (density=True, unaffected by log(y)), overlaid
    n_target, bins_target, _ = ax.hist(delta_p_target_arr, bins=30, alpha=0.55, edgecolor='black', color='deepskyblue', label='Targets', density=True)
    ax.axvline(observed_stat, color='red', linestyle='--', linewidth=3, label=f'Observed: {observed_stat:.4f}')
    ax.axvline(np.percentile(permuted_stats, 95), color='orange', linestyle=':', linewidth=2, label='95th percentile')
    ax.set_xlabel('Mean difference (permuted)', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(f'Permutation Test\n(p = {p_value_perm:.4f})', fontsize=12, fontweight='bold')
    ax.set_yscale('log')
    ax.legend()
    ax.grid(True, alpha=0.3, which='both', axis='y')
    
    # Plot 4: Effect decomposition
    ax = axes[1, 0]
    x_pos = np.arange(3)
    components = [y_target.mean(), entropy_component.mean(), targeting_component.mean()]
    labels = ['Total\nΔp(target)', 'Entropy\nComponent', 'Targeting\nComponent']
    colors = ['purple', 'orange', 'green']
    bars = ax.bar(x_pos, components, color=colors, alpha=0.7, edgecolor='black')
    ax.axhline(0, color='black', linestyle='-', linewidth=1)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels)
    ax.set_ylabel('Mean value', fontsize=11)
    ax.set_title('Effect Decomposition', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar, val in zip(bars, components):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.4f}', ha='center', va='bottom' if val > 0 else 'top', fontsize=9)
    
    # Plot 5: Scatter - entropy component vs targeting component
    ax = axes[1, 1]
    ax.scatter(entropy_component, targeting_component, alpha=0.5, s=20, c='purple')
    ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax.axvline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel('Entropy Component', fontsize=11)
    ax.set_ylabel('Targeting Component', fontsize=11)
    ax.set_title('Entropy vs Targeting Effects', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Add quadrant labels
    ax.text(0.02, 0.98, f'Both +\n({((entropy_component>0) & (targeting_component>0)).sum()} cases)', 
            transform=ax.transAxes, va='top', fontsize=8, bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
    ax.text(0.98, 0.02, f'Both -\n({((entropy_component<0) & (targeting_component<0)).sum()} cases)', 
            transform=ax.transAxes, ha='right', fontsize=8, bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.5))
    
    # Plot 6: Bootstrap distribution
    ax = axes[1, 2]
    ax.hist(bootstrap_means, bins=50, alpha=0.7, edgecolor='black', color='steelblue')
    ax.axvline(df_entropy['delta_p_target'].mean(), color='red', linestyle='--', linewidth=2, label='Observed mean')
    ax.axvline(ci_mean_lower, color='orange', linestyle=':', linewidth=2, label='95% CI')
    ax.axvline(ci_mean_upper, color='orange', linestyle=':', linewidth=2)
    ax.axvline(0, color='gray', linestyle='-', linewidth=1, alpha=0.5)
    ax.set_xlabel('Bootstrap mean Δp(target)', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(f'Bootstrap Distribution\n(CI excludes 0: {ci_mean_lower > 0})', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'statistical_tests_visualization.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nSaved statistical tests visualization to: {os.path.join(RESULTS_DIR, 'statistical_tests_visualization.png')}")